In [1]:
import importlib, subprocess, sys

def ensure(import_name, pip_name=None):
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {pip_name or import_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name], check=True)

for imp, pip in [("rdkit","rdkit"), ("xgboost","xgboost"),
                 ("lightgbm","lightgbm"), ("catboost","catboost"), ("torch","torch")]:
    ensure(imp, pip)

import os, warnings, time, numpy as np, pandas as pd
from collections import Counter
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler, PowerTransformer
from scipy.optimize import nnls
import xgboost as xgb, lightgbm as lgb
from catboost import CatBoostRegressor
import torch, torch.nn as nn, torch.nn.functional as F

from rdkit import Chem
from rdkit.Chem import Crippen, Descriptors, AllChem, MACCSkeys, rdMolDescriptors
from rdkit.Avalon import pyAvalonTools
from rdkit import DataStructs, RDLogger
RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Imports OK. Torch device:", DEVICE)

Installing rdkit ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 47.6 MB/s eta 0:00:00
Imports OK. Torch device: cuda


## 2. Data Loading & Schema Normalization

In [2]:
DATA_DIR = "/kaggle/input/competitions/ppp-round-2"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test.csv")
PI1M_PATH  = os.path.join(DATA_DIR, "PI1M.csv")

for _cand in ("train.csv", "/mnt/user-data/uploads/train.csv"):
    if not os.path.exists(TRAIN_PATH) and os.path.exists(_cand):
        TRAIN_PATH, TEST_PATH = _cand, _cand.replace("train.csv","test.csv")
for _cand in ("PI1M.csv", "/mnt/user-data/uploads/PI1M.csv"):
    if not os.path.exists(PI1M_PATH) and os.path.exists(_cand): PI1M_PATH = _cand

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

def normalize_schema(df, is_train):
    df = df.copy(); low = {c.lower(): c for c in df.columns}
    idc = next((low[c] for c in ["id","index"] if c in low), None)
    if idc is None: df.insert(0,"id",np.arange(len(df)))
    elif idc!="id": df = df.rename(columns={idc:"id"})
    sc = next((low[c] for c in ["smiles","smile","canonical_smiles"] if c in low), None)
    if sc is None: raise ValueError(f"No SMILES column in {list(df.columns)}")
    if sc!="smiles": df = df.rename(columns={sc:"smiles"})
    ttc = next((low[c] for c in ["target_type","property","property_type","task"] if c in low), None)
    if ttc and ttc!="target_type": df = df.rename(columns={ttc:"target_type"})
    if "target_type" in df.columns:
        df["target_type"] = df["target_type"].astype(str).str.strip().str.lower()
    if is_train and "target" not in df.columns:
        low = {c.lower(): c for c in df.columns}
        tc = next((low[c] for c in ["target","value","y"] if c in low), None)
        if tc!="target": df = df.rename(columns={tc:"target"})
    return df

train = normalize_schema(train, True)
test  = normalize_schema(test, False)

TARGETS = sorted(train["target_type"].unique().tolist())
print("Discovered TARGETS:", TARGETS)

_counts = train["target_type"].value_counts().to_dict()
SMALL_THRESHOLD = 600
BIG_TARGETS   = [t for t in TARGETS if _counts.get(t,0) >= SMALL_THRESHOLD]
SMALL_TARGETS = [t for t in TARGETS if _counts.get(t,0) <  SMALL_THRESHOLD]
print(f"BIG targets: {BIG_TARGETS} | SMALL targets: {SMALL_TARGETS}")

Discovered TARGETS: ['eea', 'egb', 'egc', 'ei', 'eps', 'nc', 'tg']
BIG targets: ['egc', 'tg'] | SMALL targets: ['eea', 'egb', 'ei', 'eps', 'nc']


## 2b. Canonicalization & Leak-Safe Group Folds

In [3]:
def canonical(smi):
    m = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(m) if m is not None else None

for df in (train, test):
    df["smiles_canon"] = df["smiles"].apply(canonical)
    df["smiles_canon"] = df["smiles_canon"].fillna(df["smiles"])

bad = train["smiles"].apply(lambda s: Chem.MolFromSmiles(str(s)) is None)
if bad.any(): train = train[~bad].reset_index(drop=True)

key = ["smiles_canon","target_type"]
grp = train.groupby(key)["target"]
spread = grp.transform(lambda s: s.max()-s.min())
scale = train.groupby("target_type")["target"].transform(lambda s: s.std())
bad_conflict = (spread > 0) & (spread > 0.5*scale)
train = train[~bad_conflict].reset_index(drop=True)

train["target"] = train.groupby(key)["target"].transform("median")
train = train.drop_duplicates(subset=key, keep="first").reset_index(drop=True)
train["row_id"] = np.arange(len(train)); test["row_id"]  = np.arange(len(test))

N_FOLDS = 8
def make_group_folds(groups, n_splits, seed=SEED):
    groups = np.asarray(groups)
    uniq, sizes = np.unique(groups, return_counts=True)
    rng = np.random.RandomState(seed); perm = rng.permutation(len(uniq))
    uniq, sizes = uniq[perm], sizes[perm]
    order = np.argsort(-sizes); load = np.zeros(n_splits, dtype=int); g2f={}
    for gi in order:
        f = int(np.argmin(load)); g2f[uniq[gi]] = f; load[f]+=sizes[gi]
    fid = np.array([g2f[g] for g in groups]); idx=np.arange(len(groups))
    return [(idx[fid!=f], idx[fid==f]) for f in range(n_splits)]

## 3. Featurization (With Lorentz-Lorenz Integration) & Triplet Fingerprinting

In [4]:
DESC_FUNCS = [(n,f) for n,f in Descriptors.descList]

def desc_2d(m):
    out=[]
    for _,f in DESC_FUNCS:
        try: out.append(f(m))
        except Exception: out.append(np.nan)
    return out

def morgan_counts(m, radius, nbits=1024):
    fp = AllChem.GetHashedMorganFingerprint(m, radius=radius, nBits=nbits)
    a = np.zeros(nbits, dtype=np.int16)
    for i,c in fp.GetNonzeroElements().items(): a[i]=c
    return a

def avalon_fp(m, nbits=1024):
    a = np.zeros(nbits, dtype=np.int8)
    DataStructs.ConvertToNumpyArray(pyAvalonTools.GetAvalonFP(m, nBits=nbits), a)
    return a

def maccs_fp(m):
    a = np.zeros(167, dtype=np.int8)
    DataStructs.ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(m), a)
    return a

def backbone_feats(m, smi):
    stars = [a.GetIdx() for a in m.GetAtoms() if a.GetSymbol()=="*"]
    path_len, conj_frac = np.nan, np.nan
    if len(stars) >= 2:
        try:
            p = Chem.GetShortestPath(m, stars[0], stars[1])
            if p and len(p) >= 2:
                path_len = len(p)-1
                nc = sum(1 for i in range(len(p)-1)
                         if (b:=m.GetBondBetweenAtoms(p[i],p[i+1])) is not None and b.GetIsConjugated())
                conj_frac = nc/path_len
        except Exception: pass
    na = m.GetNumAtoms()
    n_ar = sum(a.GetIsAromatic() for a in m.GetAtoms())
    n_hbd = rdMolDescriptors.CalcNumHBD(m); n_hba = rdMolDescriptors.CalcNumHBA(m)
    mw = Descriptors.MolWt(m); n_ring = rdMolDescriptors.CalcNumRings(m)
    n_rot = rdMolDescriptors.CalcNumRotatableBonds(m); fsp3 = rdMolDescriptors.CalcFractionCSP3(m)
    n_heavy = m.GetNumHeavyAtoms() or 1; tpsa = Descriptors.TPSA(m)
    
    # Calculate Molar Volume Density Proxy (Lorentz-Lorenz)
    try:
        mr = Crippen.MolMR(m)
        ll_proxy = mr / (mw / n_heavy)
    except Exception:
        ll_proxy = 0.0
        
    return [n_ar, sum(a.GetSymbol()=="O" for a in m.GetAtoms()),
            sum(a.GetSymbol()=="N" for a in m.GetAtoms()),
            sum(a.GetSymbol()=="S" for a in m.GetAtoms()),
            str(smi).count("*"), n_ar/na if na else 0.0,
            n_rot, n_ring, fsp3, path_len, conj_frac,
            n_hbd, n_hba, mw, rdMolDescriptors.CalcNumAromaticRings(m),
            (n_hbd+n_hba)/n_heavy, n_rot/n_heavy,
            n_ring/n_heavy, tpsa, tpsa/n_heavy,
            mw/(path_len+1) if not np.isnan(path_len) else np.nan,
            ll_proxy]

EXTRA = ["n_arom","n_O","n_N","n_S","n_star","arom_frac","n_rotbond",
         "n_rings","fsp3","bb_path_len","bb_conj_frac","n_hbd","n_hba","mw",
         "n_arom_ring","hbond_density","rotbond_density","ring_density",
         "tpsa","tpsa_density","mw_per_backbone","lorentz_lorenz_proxy"]

from rdkit.Chem import Fragments as _Fragments
_FRAG_FNS = [n for n in dir(_Fragments) if n.startswith("fr_")]
_GRP_SMARTS = {"amide":"C(=O)N","ester":"C(=O)O[#6]","ether":"[#6][OX2][#6]","sulfone":"S(=O)(=O)",
               "urethane":"NC(=O)O","nitrile":"C#N","ketone":"[#6]C(=O)[#6]","hydroxyl":"[OX2H]",
               "thioether":"[#6][SX2][#6]","fused_ring_hetero":"[a;R2]","imide":"C(=O)NC(=O)"}
_GRP_MOLS = {k:Chem.MolFromSmarts(v) for k,v in _GRP_SMARTS.items()}
def frag_census(m):
    if m is None: return [0.0]*(len(_FRAG_FNS)+len(_GRP_MOLS))
    out=[]
    for fn in _FRAG_FNS:
        try: out.append(float(getattr(_Fragments,fn)(m)))
        except Exception: out.append(0.0)
    for k in _GRP_MOLS:
        p=_GRP_MOLS[k]; out.append(float(len(m.GetSubstructMatches(p))) if p is not None else 0.0)
    return out
FRAG_COLS = [f"fr_{n}" for n in _FRAG_FNS] + [f"grp_{k}" for k in _GRP_MOLS]

from rdkit.Chem.AtomPairs import Pairs as _Pairs, Torsions as _Torsions
AP_BITS=512; TT_BITS=512; RDKFP_BITS=1024
def atompair_counts(m,n=AP_BITS):
    v=np.zeros(n,np.int16)
    if m is None: return v
    try:
        fp=_Pairs.GetHashedAtomPairFingerprint(m,nBits=n)
        for i,c in fp.GetNonzeroElements().items(): v[i%n]+=c
    except Exception: pass
    return v
def torsion_counts(m,n=TT_BITS):
    v=np.zeros(n,np.int16)
    if m is None: return v
    try:
        fp=_Torsions.GetHashedTopologicalTorsionFingerprint(m,nBits=n)
        for i,c in fp.GetNonzeroElements().items(): v[i%n]+=c
    except Exception: pass
    return v
def rdkfp_bits(m,n=RDKFP_BITS):
    v=np.zeros(n,np.int8)
    if m is None: return v
    try:
        fp=Chem.RDKFingerprint(m,fpSize=n,maxPath=7); DataStructs.ConvertToNumpyArray(fp,v)
    except Exception: pass
    return v
def fluor_density(m):
    if m is None: return [0.0]*4
    na=m.GetNumHeavyAtoms() or 1; nF=sum(1 for a in m.GetAtoms() if a.GetSymbol()=="F")
    nC=sum(1 for a in m.GetAtoms() if a.GetSymbol()=="C") or 1
    fC=sum(1 for a in m.GetAtoms() if a.GetSymbol()=="C" and any(nb.GetSymbol()=="F" for nb in a.GetNeighbors()))
    cf23=sum(1 for a in m.GetAtoms() if a.GetSymbol()=="C" and sum(nb.GetSymbol()=="F" for nb in a.GetNeighbors())>=2)
    return [nF/na, fC/nC, cf23/nC, float(nF)]
FLUOR_COLS=["x_Fdens","x_fluorC_frac","x_CF23_frac","x_Fcount"]
_SULFUR_SMARTS={"thiophene":"c1ccsc1","thiocarbonyl":"[CX3]=[SX1]","CS_any":"C=S","thioamide":"[NX3]C(=[SX1])","thioether":"[#6][SX2][#6]"}
_SULFUR_MOLS={k:Chem.MolFromSmarts(v) for k,v in _SULFUR_SMARTS.items()}
SULFUR_COLS=["x_"+k for k in _SULFUR_SMARTS]
def sulfur_motifs(m):
    if m is None: return [0.0]*len(_SULFUR_MOLS)
    na=m.GetNumHeavyAtoms() or 1
    return [len(m.GetSubstructMatches(_SULFUR_MOLS[k]))/na if _SULFUR_MOLS[k] else 0.0 for k in _SULFUR_MOLS]

def featurize(df):
    D,M2,M3,AV,MA,EX,FR,AP,TT,RK,FL,SU = [],[],[],[],[],[],[],[],[],[],[],[]
    for s in df["smiles"]:
        m = Chem.MolFromSmiles(str(s))
        if m is None:
            D.append([np.nan]*len(DESC_FUNCS)); M2.append(np.zeros(1024,np.int16))
            M3.append(np.zeros(1024,np.int16)); AV.append(np.zeros(1024,np.int8))
            MA.append(np.zeros(167,np.int8)); EX.append([np.nan]*len(EXTRA)); FR.append([0.0]*len(FRAG_COLS)); AP.append(np.zeros(AP_BITS,np.int16)); TT.append(np.zeros(TT_BITS,np.int16)); RK.append(np.zeros(RDKFP_BITS,np.int8)); FL.append([0.0]*len(FLUOR_COLS)); SU.append([0.0]*len(SULFUR_COLS)); continue
        D.append(desc_2d(m)); M2.append(morgan_counts(m,2,1024))
        M3.append(morgan_counts(m,3,1024)); AV.append(avalon_fp(m)); MA.append(maccs_fp(m))
        EX.append(backbone_feats(m,s)); FR.append(frag_census(m)); AP.append(atompair_counts(m)); TT.append(torsion_counts(m)); RK.append(rdkfp_bits(m)); FL.append(fluor_density(m)); SU.append(sulfur_motifs(m))
    dcol=[f"d_{n}" for n,_ in DESC_FUNCS]; excol=[f"x_{c}" for c in EXTRA]
    m2col=[f"m2_{i}" for i in range(1024)]; m3col=[f"m3_{i}" for i in range(1024)]
    avcol=[f"av_{i}" for i in range(1024)]; macol=[f"ma_{i}" for i in range(167)]
    apcol=[f"ap_{i}" for i in range(AP_BITS)]; ttcol=[f"tt_{i}" for i in range(TT_BITS)]; rkcol=[f"rk_{i}" for i in range(RDKFP_BITS)]
    X = np.hstack([np.array(D,dtype=np.float32), np.array(EX,dtype=np.float32),
                   np.array(FL,dtype=np.float32), np.array(SU,dtype=np.float32),
                   np.array(AP), np.array(TT), np.array(RK),
                   np.array(FR,dtype=np.float32),
                   np.array(M2), np.array(M3), np.array(AV), np.array(MA)])
    cols = dcol+excol+FLUOR_COLS+SULFUR_COLS+apcol+ttcol+rkcol+FRAG_COLS+m2col+m3col+avcol+macol
    return pd.DataFrame(X, columns=cols), dcol+excol+FLUOR_COLS+SULFUR_COLS+FRAG_COLS, apcol+ttcol+rkcol+m2col+m3col+avcol+macol

print("Featurizing dataset ...")
Xtr_df, DENSE_COLS, SPARSE_COLS = featurize(train)
Xte_df, _, _ = featurize(test)

ALL_COLS = DENSE_COLS + SPARSE_COLS; CLIP = 1e6
def sanitize(df, cols, med=None):
    df = df.copy(); df[cols] = df[cols].replace([np.inf,-np.inf], np.nan)
    if med is None: med = df[cols].median().fillna(0.0)
    df[cols] = df[cols].fillna(med).fillna(0.0).clip(-CLIP, CLIP)
    return df, med

Xtr_df, med = sanitize(Xtr_df, DENSE_COLS)
Xte_df, _   = sanitize(Xte_df, DENSE_COLS, med)

dv = Xtr_df[DENSE_COLS].var(); DENSE_COLS=[c for c in DENSE_COLS if dv[c]>0]
sv = Xtr_df[SPARSE_COLS].var(); SPARSE_COLS=[c for c in SPARSE_COLS if sv[c]>1e-4]
ALL_COLS = DENSE_COLS + SPARSE_COLS

Xtr_all = Xtr_df[ALL_COLS].values.astype(np.float32)
Xte_all = Xte_df[ALL_COLS].values.astype(np.float32)

# ===== Kuenneth Level-1 Triplets =====
def _atom_token(atom): return f'{atom.GetSymbol()}{atom.GetDegree()}'
def _get_triplets(mol):
    if mol is None: return []
    triplets = []
    for bond in mol.GetBonds():
        j, k = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        atom_j, atom_k = mol.GetAtomWithIdx(j), mol.GetAtomWithIdx(k)
        for nb in atom_j.GetNeighbors():
            if nb.GetIdx() == k: continue
            triplets.append((_atom_token(nb), _atom_token(atom_j), _atom_token(atom_k)))
        for nb in atom_k.GetNeighbors():
            if nb.GetIdx() == j: continue
            triplets.append((_atom_token(atom_j), _atom_token(atom_k), _atom_token(nb)))
    return triplets

_vocab_counter = Counter()
for smi in train['smiles'].values:
    _vocab_counter.update(_get_triplets(Chem.MolFromSmiles(str(smi))))

_TRIPLET_VOCAB = sorted([t for t, c in _vocab_counter.items() if c >= 5])
_TRIPLET_IDX   = {t: i for i, t in enumerate(_TRIPLET_VOCAB)}

def _triplet_fp(smi):
    mol = Chem.MolFromSmiles(str(smi))
    vec = np.zeros(len(_TRIPLET_VOCAB), dtype=np.float32)
    if mol is None: return vec
    for t in _get_triplets(mol):
        if t in _TRIPLET_IDX: vec[_TRIPLET_IDX[t]] += 1.0
    return vec

TRP_tr = np.array([_triplet_fp(s) for s in train['smiles'].values], dtype=np.float32)
TRP_te = np.array([_triplet_fp(s) for s in test['smiles'].values],  dtype=np.float32)

Xtr_all = np.hstack([Xtr_all, TRP_tr]).astype(np.float32)
Xte_all = np.hstack([Xte_all, TRP_te]).astype(np.float32)
ALL_COLS = ALL_COLS + [f'triplet_{i}' for i in range(len(_TRIPLET_VOCAB))]
print(f"Featurization done: total {Xtr_all.shape[1]} features")

Featurizing dataset ...
Featurization done: total 6204 features


## 3b. Domain Physics Injection & Cross-Target Lookups

In [5]:
_canon_tr = train["smiles_canon"].values; _tt_tr = train["target_type"].values
_y_tr = train["target"].values.astype(np.float32)
_canon_te = test["smiles_canon"].values;  _tt_te = test["target_type"].values

_LUT = {t: {} for t in TARGETS}
for c,t,v in zip(_canon_tr,_tt_tr,_y_tr): _LUT[t][c]=v
_MED = {t:(float(np.median(list(_LUT[t].values()))) if _LUT[t] else 0.0) for t in TARGETS}

def _cross_block(canon):
    n=len(canon); T=len(TARGETS)
    vals=np.zeros((n,T),np.float32); known=np.zeros((n,T),np.float32)
    for j,t in enumerate(TARGETS):
        lut=_LUT[t]
        for i2,c in enumerate(canon):
            if c in lut: vals[i2,j]=lut[c]; known[i2,j]=1.0
            else:        vals[i2,j]=_MED[t]
    return vals,known

CROSS_tr_val,CROSS_tr_known = _cross_block(_canon_tr)
CROSS_te_val,CROSS_te_known = _cross_block(_canon_te)

def _mk_cross(): return lgb.LGBMRegressor(n_estimators=500,learning_rate=0.03,num_leaves=63,
    subsample=0.85,colsample_bytree=0.4,reg_lambda=1.0,random_state=SEED,n_jobs=-1,verbosity=-1)

for fi_,(tr,va) in enumerate(make_group_folds(_canon_tr,N_FOLDS,seed=SEED)):
    for j,t in enumerate(TARGETS):
        need=va[(CROSS_tr_known[va,j]==0.0)]
        if len(need)==0: continue
        src=tr[_tt_tr[tr]==t]
        if len(src)<30: continue
        m=_mk_cross().fit(Xtr_all[src],_y_tr[src]); CROSS_tr_val[need,j]=m.predict(Xtr_all[need])

_full_cross={}
for t in TARGETS:
    src=np.where(_tt_tr==t)[0]
    if len(src)>=30: _full_cross[t]=_mk_cross().fit(Xtr_all[src],_y_tr[src])
for j,t in enumerate(TARGETS):
    if t not in _full_cross: continue
    need=np.where(CROSS_te_known[:,j]==0.0)[0]
    if len(need): CROSS_te_val[need,j]=_full_cross[t].predict(Xte_all[need])

_self={t:j for j,t in enumerate(TARGETS)}
for i2 in range(len(_tt_tr)):
    j=_self[_tt_tr[i2]]; CROSS_tr_val[i2,j]=_MED[TARGETS[j]]; CROSS_tr_known[i2,j]=0.0
for i2 in range(len(_tt_te)):
    j=_self[_tt_te[i2]]; CROSS_te_val[i2,j]=_MED[TARGETS[j]]; CROSS_te_known[i2,j]=0.0

# Physical Identities (Koopmans & Maxwell)
_ti={t:i for i,t in enumerate(TARGETS)}
def _col(mat,t): return mat[:,_ti[t]] if t in _ti else np.zeros(mat.shape[0],np.float32)

def _physics_block(cross_val, cross_known):
    v=lambda t:_col(cross_val,t); k=lambda t:_col(cross_known,t)
    feats={}
    if all(t in _ti for t in ["eea","egc"]):
        feats["phys_ei_koop"]      = v("eea")+v("egc")
        feats["phys_ei_koop_conf"] = (v("eea")+v("egc"))*k("eea")*k("egc")
    if all(t in _ti for t in ["ei","egc"]):
        feats["phys_eea_koop"]     = v("ei")-v("egc")
        feats["phys_eea_koop_conf"]= (v("ei")-v("egc"))*k("ei")*k("egc")
    if all(t in _ti for t in ["ei","eea"]):
        feats["phys_egc_koop"]     = v("ei")-v("eea")
        feats["phys_egc_koop_conf"]= (v("ei")-v("eea"))*k("ei")*k("eea")
    if "nc" in _ti:
        feats["phys_eps_n2"]       = v("nc")**2
        feats["phys_eps_n2_conf"]  = (v("nc")**2)*k("nc")
    if all(t in _ti for t in ["nc","egc"]):
        feats["phys_eps_polar"]    = (v("nc")**2 - 1.0)/np.clip(v("nc")**2 + 2.0,1e-3,None)
        feats["phys_eps_invgap"]   = 1.0/np.clip(v("egc"),0.1,None)
    names=list(feats.keys())
    M=np.column_stack([feats[n] for n in names]).astype(np.float32)
    return np.nan_to_num(M,nan=0.0,posinf=0.0,neginf=0.0), names

PHYS_tr,_pn = _physics_block(CROSS_tr_val, CROSS_tr_known)
PHYS_te,_   = _physics_block(CROSS_te_val, CROSS_te_known)

Xtr_all=np.hstack([Xtr_all, CROSS_tr_val, CROSS_tr_known, PHYS_tr]).astype(np.float32)
Xte_all=np.hstack([Xte_all, CROSS_te_val, CROSS_te_known, PHYS_te]).astype(np.float32)
ALL_COLS=ALL_COLS+[f"cross_{t}" for t in TARGETS]+[f"cross_{t}_known" for t in TARGETS]+_pn
DENSE_COLS=DENSE_COLS+[f"cross_{t}" for t in TARGETS]+[f"cross_{t}_known" for t in TARGETS]+_pn
print(f"Physics and cross-target features added: total {Xtr_all.shape[1]} features")

Physics and cross-target features added: total 6228 features


## 3c. Dual-Tier Feature Selection (Strict Capping for Small N)

### 3j2. EHT quantum features for ei ONLY (crash-isolated)

Extended Huckel Theory gives real quantum-mechanical orbital energies from a 3D-embedded structure —
measured +0.062 R2 on ei (the biggest single feature gain in this project), essentially flat/negative on
eps/nc, so restricted to ei's ~370 molecules (train+test). `AllChem.EmbedMolecule` + `rdEHTTools.RunMol`
can segfault the interpreter on some inputs (not a catchable Python exception), which would kill a 9-hour
Kaggle run outright. Each molecule runs in an **isolated worker subprocess with a hard timeout**; a crash
or hang in a worker cannot bring down the main kernel — that row just falls back to zeros. If the whole
subprocess pool is unavailable for any reason, the block degrades to all-zero EHT columns rather than
failing the notebook.

In [6]:
# ===== EHT quantum features for ei only (process-isolated, timeout-guarded) =====
EHT_ENABLE = False   # measured ~0 gain on ei (lgb 0.8734 with vs without); costs runtime and floods the log with geometry warnings
EHT_TIMEOUT_S = 20        # per-molecule hard timeout
EHT_COLS = ["x_eht_homo","x_eht_lumo","x_eht_gap","x_eht_fermi","x_eht_etot_peratom"]

def _eht_worker(smi):
    # runs in a CHILD process; a segfault here only kills the worker, not the main kernel
    from rdkit import Chem as _Chem
    from rdkit.Chem import AllChem as _AllChem, rdEHTTools as _rdEHTTools
    import numpy as _np
    m = _Chem.MolFromSmiles(str(smi).replace("*","[H]"))
    if m is None: return [0.0]*5
    m = _Chem.AddHs(m)
    try:
        cid = _AllChem.EmbedMolecule(m, randomSeed=42, maxAttempts=20)
        if cid < 0: return [0.0]*5
        _AllChem.MMFFOptimizeMolecule(m, maxIters=200)
        ok, res = _rdEHTTools.RunMol(m)
        if not ok: return [0.0]*5
        E = _np.array(res.GetOrbitalEnergies()); ne = res.numElectrons; hi = ne//2 - 1
        homo = float(E[hi]) if 0 <= hi < len(E) else 0.0
        lumo = float(E[hi+1]) if hi+1 < len(E) else 0.0
        return [homo, lumo, lumo-homo, float(res.fermiEnergy), float(res.totalEnergy)/max(m.GetNumAtoms(),1)]
    except Exception:
        return [0.0]*5

def compute_eht_for_target(smiles_list, target_name, timeout_s=EHT_TIMEOUT_S):
    import concurrent.futures as _cf
    out = [[0.0]*5 for _ in smiles_list]
    if not EHT_ENABLE or len(smiles_list) == 0:
        return np.array(out, dtype=np.float32)
    try:
        with _cf.ProcessPoolExecutor(max_workers=2) as ex:
            futs = {ex.submit(_eht_worker, s): i for i, s in enumerate(smiles_list)}
            for fut in _cf.as_completed(futs):
                i = futs[fut]
                try:
                    out[i] = fut.result(timeout=timeout_s)
                except Exception:
                    out[i] = [0.0]*5   # crash / timeout / broken worker -> safe zero fallback
        n_nonzero = sum(1 for r in out if any(v != 0.0 for v in r))
        print(f"EHT ({target_name}): {n_nonzero}/{len(smiles_list)} molecules succeeded")
    except Exception as e:
        print(f"EHT pool unavailable ({target_name}), falling back to zeros:", repr(e))
    return np.array(out, dtype=np.float32)

# Inject EHT as extra columns ONLY on ei rows (train + test); zeros elsewhere
_eht_extra_tr = np.zeros((Xtr_all.shape[0], len(EHT_COLS)), dtype=np.float32)
_eht_extra_te = np.zeros((Xte_all.shape[0], len(EHT_COLS)), dtype=np.float32)
if EHT_ENABLE and "ei" in TARGETS:
    _ei_tr_mask = (train["target_type"].values == "ei")
    _ei_te_mask = (test["target_type"].values == "ei")
    if _ei_tr_mask.any():
        _eht_extra_tr[_ei_tr_mask] = compute_eht_for_target(train.loc[_ei_tr_mask,"smiles"].tolist(), "ei-train")
    if _ei_te_mask.any():
        _eht_extra_te[_ei_te_mask] = compute_eht_for_target(test.loc[_ei_te_mask,"smiles"].tolist(), "ei-test")

Xtr_all = np.hstack([Xtr_all, _eht_extra_tr]).astype(np.float32)
Xte_all = np.hstack([Xte_all, _eht_extra_te]).astype(np.float32)
ALL_COLS = ALL_COLS + EHT_COLS
print(f"EHT columns added -> total {Xtr_all.shape[1]} features (nonzero only on ei rows)")


EHT columns added -> total 6233 features (nonzero only on ei rows)


In [7]:
TOP_K = 900
SMALL_K = 250


def select_features(target):
    """Per-target importance selection.

    Change vs previous version: for the SMALL targets a single LightGBM importance probe fit
    on ~220 rows is itself high-variance -- which features land in the top-K shifts noticeably
    between reruns. Here the probe is refit across the group folds and feature *ranks* are
    averaged, so a column has to look useful consistently rather than once. Big targets keep
    the single fit (4139/2028 rows is enough for a stable ranking, and this keeps runtime down).
    """
    mask = (train["target_type"].values == target)
    X = Xtr_all[mask]
    y = train.loc[mask, "target"].values.astype(np.float32)
    groups = train.loc[mask, "smiles_canon"].values
    is_small = target in SMALL_TARGETS
    k = SMALL_K if is_small else TOP_K
    nl = 15 if is_small else 63

    def _probe(seed):
        return lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=nl,
                                 subsample=0.8, colsample_bytree=0.6,
                                 random_state=seed, n_jobs=-1, verbosity=-1)

    if is_small:
        rank_sum = np.zeros(X.shape[1], dtype=np.float64)
        n_used = 0
        for tr, _va in make_group_folds(groups, min(5, N_FOLDS), seed=SEED):
            p = _probe(SEED).fit(X[tr], y[tr])
            # rank 0 == most important; summing ranks rewards consistency across folds
            rank_sum += np.argsort(np.argsort(-p.feature_importances_))
            n_used += 1
        score = -rank_sum / max(n_used, 1)
    else:
        score = _probe(SEED).fit(X, y).feature_importances_.astype(np.float64)

    top_idx = np.argsort(-score)[:min(k, len(score))]
    # Force-protect physics / cross-target / backbone columns. Tree split-gain systematically
    # under-ranks smooth linear identities (Koopmans sums, n^2, Clausius-Mossotti), so they can
    # be pruned despite carrying real signal -- measured harmful on the small targets.
    _force = [j for j, cc in enumerate(ALL_COLS) if str(cc).startswith(("phys_", "cross_", "x_"))]
    combined = np.union1d(top_idx, np.array(_force, dtype=int)) if _force else top_idx
    return np.sort(combined)


SELECTED = {t: select_features(t) for t in TARGETS}
for t in TARGETS:
    _tag = "fold-bagged" if t in SMALL_TARGETS else "single-fit"
    print(f"{t}: kept {len(SELECTED[t])}/{len(ALL_COLS)} features  ({_tag} ranking)")


eea: kept 279/6233 features  (fold-bagged ranking)
egb: kept 277/6233 features  (fold-bagged ranking)
egc: kept 927/6233 features  (single-fit ranking)
ei: kept 277/6233 features  (fold-bagged ranking)
eps: kept 277/6233 features  (fold-bagged ranking)
nc: kept 276/6233 features  (fold-bagged ranking)
tg: kept 928/6233 features  (single-fit ranking)


## 3d. PI1M Cross-Task Pseudo-Labeling (Ei and Eea)

In [8]:
PSEUDO_N = 5000; PSEUDO_W = 0.2; PSEUDO = {}

if os.path.exists(PI1M_PATH) and ("ei" in TARGETS or "eea" in TARGETS):
    try:
        _pdf = pd.read_csv(PI1M_PATH)
        _pcol = "SMILES" if "SMILES" in _pdf.columns else _pdf.columns[0]
        _pick = np.random.RandomState(SEED).permutation(len(_pdf))[:PSEUDO_N*3]
        _psmi = _pdf.iloc[_pick][_pcol].astype(str).values
        _pf_df, _, _ = featurize(pd.DataFrame({"smiles": _psmi}))
        _col_pos = {c:i for i,c in enumerate(ALL_COLS)}
        _Xp = np.zeros((len(_psmi), len(ALL_COLS)), dtype=np.float32)
        for c in _pf_df.columns:
            if c in _col_pos:
                _Xp[:, _col_pos[c]] = pd.to_numeric(_pf_df[c], errors="coerce").fillna(0.0).values.astype(np.float32)
        _Xp = np.nan_to_num(_Xp)
        _valid = (_Xp.sum(1)!=0)

        def _fit_full(tgt):
            m = (train["target_type"].values==tgt)
            mdl = lgb.LGBMRegressor(n_estimators=700, learning_rate=0.03, num_leaves=31,
                     subsample=0.8, colsample_bytree=0.5, random_state=SEED, n_jobs=-1, verbosity=-1)
            mdl.fit(Xtr_all[m][:, SELECTED[tgt]], train.loc[m,"target"].values.astype(float))
            return mdl
        _m = {t:_fit_full(t) for t in ["egc","eea","ei"] if t in TARGETS}
        def _range(tgt): v = train.loc[train.target_type==tgt,"target"].values.astype(float); return v.min(), v.max()

        if "ei" in TARGETS and "egc" in _m and "eea" in _m:
            yp = _m["egc"].predict(_Xp[:, SELECTED["egc"]]) + _m["eea"].predict(_Xp[:, SELECTED["eea"]])
            lo,hi = _range("ei"); ok = _valid & (yp>=lo) & (yp<=hi)
            sel = np.where(ok)[0][:PSEUDO_N]
            PSEUDO["ei"] = (_Xp[sel].copy(), yp[sel].astype(np.float32))
        if "eea" in TARGETS and "ei" in _m and "egc" in _m:
            yp = _m["ei"].predict(_Xp[:, SELECTED["ei"]]) - _m["egc"].predict(_Xp[:, SELECTED["egc"]])
            lo,hi = _range("eea"); ok = _valid & (yp>=lo) & (yp<=hi)
            sel = np.where(ok)[0][:PSEUDO_N]
            PSEUDO["eea"] = (_Xp[sel].copy(), yp[sel].astype(np.float32))
        print(f"Pseudo-labeling generated: " + ", ".join(f"{t}:{len(PSEUDO[t][1])} rows" for t in PSEUDO))
    except Exception as e:
        print("Pseudo-labeling skipped:", repr(e))
else:
    print("Pseudo-labeling skipped (PI1M.csv not found)")

Pseudo-labeling generated: ei:5000 rows, eea:5000 rows


## 4. Seed-Bagged Tree Ensembles (1 Seed for Fast Execution)

In [9]:
class _RawT:
    def fit_transform(self,y): return y.ravel()
    def transform(self,y): return y.ravel()
    def inverse_transform(self,y): return y.ravel()
class _CMTransform:
    # Clausius-Mossotti bounded transform for eps: (e-1)/(e+2) in [0,1). Measured +0.017 R2 on eps
    # (tames the heavy right tail from the sulfur-cluster outlier group) vs raw/yeo-johnson.
    def fit_transform(self,y): return ((y.ravel()-1.0)/(y.ravel()+2.0))
    def transform(self,y): return ((y.ravel()-1.0)/(y.ravel()+2.0))
    def inverse_transform(self,y):
        t=np.clip(y.ravel(),0.0,0.999); return ((1.0+2.0*t)/(1.0-t))
def _make_t(mode):
    if mode=="yj": return PowerTransformer(method="yeo-johnson")
    if mode=="cm": return _CMTransform()
    return _RawT()

def get_models(seed):
    return {
        "xgb": xgb.XGBRegressor(n_estimators=3000, learning_rate=0.02, max_depth=6,
                subsample=0.8, colsample_bytree=0.3, colsample_bylevel=0.5, reg_alpha=0.2,
                reg_lambda=1.5, min_child_weight=3, random_state=seed, n_jobs=-1, tree_method="hist",
                early_stopping_rounds=120, eval_metric="rmse"),
        "lgb": lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.02, num_leaves=63,
                subsample=0.8, subsample_freq=1, colsample_bytree=0.3, reg_alpha=0.2,
                reg_lambda=1.5, min_child_samples=20, random_state=seed, n_jobs=-1, verbosity=-1),
        "lgb_et": lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.02, num_leaves=127,
                subsample=0.7, subsample_freq=1, colsample_bytree=0.25, reg_alpha=0.1, reg_lambda=1.0,
                min_child_samples=10, extra_trees=True, random_state=seed+7, n_jobs=-1, verbosity=-1),
        "cat": CatBoostRegressor(iterations=3000, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
                rsm=0.3, random_seed=seed, verbose=False, early_stopping_rounds=120),
    }

def get_models_small(seed):
    return {
        "cat": CatBoostRegressor(iterations=1500, learning_rate=0.03, depth=4, l2_leaf_reg=8.0,
                rsm=0.3, random_seed=seed, verbose=False, early_stopping_rounds=80),
        "lgb": lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.03, num_leaves=15,
                subsample=0.8, subsample_freq=1, colsample_bytree=0.3, reg_alpha=0.5,
                reg_lambda=3.0, min_child_samples=8, random_state=seed, n_jobs=-1, verbosity=-1),
    }

TREE_SEEDS = [SEED, SEED+1, SEED+2]  # bagged (was single-seed; 3x not 5x to respect GPU runtime budget)

def train_trees(target):
    mask = (train["target_type"].values == target)
    feat_idx = SELECTED[target]
    X, y = Xtr_all[mask][:, feat_idx], train.loc[mask,"target"].values.astype(np.float32)
    Xt = Xte_all[:, feat_idx]
    groups = train.loc[mask,"smiles_canon"].values
    is_small = target in SMALL_TARGETS
    names = ["cat","lgb"] if is_small else ["xgb","lgb","lgb_et","cat"]
    tmap  = {"cat":"yj","lgb":"yj"} if is_small else {"xgb":"yj","lgb":"raw","lgb_et":"yj","cat":"yj"}
    # eps: Clausius-Mossotti bounded transform beats yeo-johnson/raw (measured +0.017 R2)
    if target == "eps":
        tmap = {n:"cm" for n in tmap}
    mkfn  = get_models_small if is_small else get_models
    oof = {n: np.zeros(len(y)) for n in names}
    tst = {n: np.zeros(len(test)) for n in names}
    for seed in TREE_SEEDS:
        for tr,va in make_group_folds(groups, N_FOLDS, seed=seed):
            mdl = mkfn(seed)
            _has_ps = (target in PSEUDO) and (len(PSEUDO[target][1])>0)
            if _has_ps:
                _Xps = PSEUDO[target][0][:, feat_idx]; _yps = PSEUDO[target][1]
                _sw = np.concatenate([np.ones(len(tr), dtype=np.float32), np.full(len(_yps), PSEUDO_W, dtype=np.float32)])
            for n in names:
                pt  = _make_t(tmap[n])
                ytr = pt.fit_transform(y[tr].reshape(-1,1)).ravel()
                yva = pt.transform(y[va].reshape(-1,1)).ravel()
                if _has_ps:
                    Xtr_aug = np.vstack([X[tr], _Xps]); ytr_aug = np.concatenate([ytr, pt.transform(_yps.reshape(-1,1)).ravel()])
                else:
                    Xtr_aug = X[tr]; ytr_aug = ytr
                if n.startswith("xgb"):
                    mdl[n].fit(Xtr_aug,ytr_aug,sample_weight=(_sw if _has_ps else None),eval_set=[(X[va],yva)],verbose=False)
                elif n.startswith("lgb"):
                    mdl[n].fit(Xtr_aug,ytr_aug,sample_weight=(_sw if _has_ps else None),eval_set=[(X[va],yva)],callbacks=[lgb.early_stopping(120,verbose=False)])
                else:
                    mdl[n].fit(Xtr_aug,ytr_aug,sample_weight=(_sw if _has_ps else None),eval_set=(X[va],yva))
                oof[n][va] += pt.inverse_transform(mdl[n].predict(X[va]).reshape(-1,1)).ravel() / len(TREE_SEEDS)
                tst[n]     += pt.inverse_transform(mdl[n].predict(Xt).reshape(-1,1)).ravel() / (len(TREE_SEEDS)*N_FOLDS)
    blend = np.mean([oof[n] for n in names], axis=0)
    print(f"[{target}] tree blend OOF R2 = {r2_score(y,blend):.4f}")
    return {"oof":oof, "test":tst, "y":y, "mask":mask}

tree_res = {t: train_trees(t) for t in TARGETS}

[eea] tree blend OOF R2 = 0.9178
[egb] tree blend OOF R2 = 0.9407
[egc] tree blend OOF R2 = 0.9134
[ei] tree blend OOF R2 = 0.8727
[eps] tree blend OOF R2 = 0.8196
[nc] tree blend OOF R2 = 0.8897
[tg] tree blend OOF R2 = 0.9194


## 4b. Ridge Regression & Gaussian Process Regression (GPR)

In [10]:
def train_ridge(target):
    mask = (train["target_type"].values == target)
    dense_idx = [i for i,c in enumerate(ALL_COLS) if c in DENSE_COLS]
    X, y = Xtr_all[mask][:, dense_idx], train.loc[mask,"target"].values.astype(np.float32)
    Xt = Xte_all[:, dense_idx]
    groups = train.loc[mask,"smiles_canon"].values
    oof = np.zeros(len(y)); tst = np.zeros(len(test))
    for tr,va in make_group_folds(groups, N_FOLDS, seed=SEED):
        sc = StandardScaler().fit(X[tr])
        model = Ridge(alpha=5.0).fit(sc.transform(X[tr]), y[tr])
        oof[va] = model.predict(sc.transform(X[va]))
        tst += model.predict(sc.transform(Xt)) / N_FOLDS
    print(f"[{target}] ridge OOF R2 = {r2_score(y,oof):.4f}")
    return {"oof":oof, "test":tst}

ridge_res = {t: train_ridge(t) for t in TARGETS}


[eea] ridge OOF R2 = 0.8929
[egb] ridge OOF R2 = 0.8883
[egc] ridge OOF R2 = 0.8573
[ei] ridge OOF R2 = 0.7667
[eps] ridge OOF R2 = 0.8043
[nc] ridge OOF R2 = 0.8795
[tg] ridge OOF R2 = 0.8346


## 5. Pure PyTorch Multi-Task GNN

In [11]:
ATOM_LIST = ["C","N","O","S","F","Si","P","Cl","Br","I","B","H","*"]
ATOM_MAP  = {s:i for i,s in enumerate(ATOM_LIST)}
HYB = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
       Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
       Chem.rdchem.HybridizationType.SP3D2]
HYB_MAP = {h:i for i,h in enumerate(HYB)}
BONDS = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
         Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
BT_MAP = {b:i for i,b in enumerate(BONDS)}

def _onehot(i,n):
    v=[0.0]*n
    if i is not None and 0<=i<n: v[i]=1.0
    return v
def atom_feat(a):
    return (_onehot(ATOM_MAP.get(a.GetSymbol()), len(ATOM_LIST)) + _onehot(min(a.GetDegree(),5),6)
            + _onehot(min(a.GetTotalNumHs(),4),5) + _onehot(HYB_MAP.get(a.GetHybridization()),len(HYB))
            + [float(a.GetIsAromatic()), float(a.IsInRing()),
               float(a.GetFormalCharge()), float(a.GetSymbol()=="*")])
def bond_feat(b):
    return _onehot(BT_MAP.get(b.GetBondType()),len(BONDS)) + [float(b.GetIsConjugated()), float(b.IsInRing())]
_m = Chem.MolFromSmiles("CC"); ADIM=len(atom_feat(_m.GetAtomWithIdx(0))); BDIM=len(bond_feat(_m.GetBondWithIdx(0)))

def to_graph(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None or m.GetNumAtoms()==0: return None
    x = np.array([atom_feat(a) for a in m.GetAtoms()], dtype=np.float32)
    s,d,e = [],[],[]
    for b in m.GetBonds():
        i,j=b.GetBeginAtomIdx(),b.GetEndAtomIdx(); bf=bond_feat(b)
        s+=[i,j]; d+=[j,i]; e+=[bf,bf]
    if not s: s,d,e=[0],[0],[[0.0]*BDIM]
    return x, np.array([s,d],dtype=np.int64), np.array(e,dtype=np.float32)

G_tr = [to_graph(s) for s in train["smiles"]]
G_te = [to_graph(s) for s in test["smiles"]]

def collate(graphs):
    xs,eis,eas,batch=[],[],[],[]; off=0
    for gi,g in enumerate(graphs):
        x,ei,ea=g; xs.append(x); eis.append(ei+off); eas.append(ea)
        batch += [gi]*x.shape[0]; off += x.shape[0]
    return (torch.tensor(np.concatenate(xs)),
            torch.tensor(np.concatenate(eis,axis=1)),
            torch.tensor(np.concatenate(eas)),
            torch.tensor(batch,dtype=torch.int64))

class MultiTaskMPNN(nn.Module):
    def __init__(self, adim, bdim, hid=192, layers=4, drop=0.1, n_tasks=len(TARGETS)):
        super().__init__(); self.L=layers
        self.lin0=nn.Linear(adim,hid); self.edge=nn.Linear(bdim,hid)
        self.msg=nn.ModuleList([nn.Linear(2*hid,hid) for _ in range(layers)])
        self.upd=nn.ModuleList([nn.GRUCell(hid,hid) for _ in range(layers)])
        self.bn =nn.ModuleList([nn.BatchNorm1d(hid) for _ in range(layers)])
        self.attn_w=nn.Linear(hid,1)   # attention-weighted pooling channel (measured +0.019 ei, +0.002 egc)
        self.trunk=nn.Sequential(nn.Linear(3*hid,hid),nn.ReLU(),nn.Dropout(drop))
        self.heads=nn.ModuleList([nn.Sequential(nn.Linear(hid,hid//2),nn.ReLU(),nn.Linear(hid//2,1))
                                  for _ in range(n_tasks)])
    def forward(self,X,EI,EA,B):
        h=F.relu(self.lin0(X)); e=self.edge(EA); s,d=EI[0],EI[1]
        for l in range(self.L):
            msg=torch.relu(self.msg[l](torch.cat([h[s],e],1)))
            agg=torch.zeros_like(h).index_add_(0,d,msg)
            h=self.bn[l](self.upd[l](agg,h))
        ng=int(B.max().item())+1
        summ=torch.zeros(ng,h.size(1),device=h.device).index_add_(0,B,h)
        cnt=torch.zeros(ng,1,device=h.device).index_add_(0,B,torch.ones(h.size(0),1,device=h.device))
        mean=summ/cnt.clamp(min=1)
        mx=torch.full((ng,h.size(1)),-1e9,device=h.device).index_reduce_(0,B,h,"amax",include_self=True)
        # attention-weighted pooling: softmax per-graph over atom scores, weighted sum
        _scores=self.attn_w(h).squeeze(-1)
        _exp=torch.exp(_scores-_scores.max())
        _denom=torch.zeros(ng,device=h.device).index_add_(0,B,_exp).clamp(min=1e-8)
        _w=(_exp/_denom[B]).unsqueeze(-1)
        attn_pool=torch.zeros(ng,h.size(1),device=h.device).index_add_(0,B,h*_w)
        z = self.trunk(torch.cat([mean,mx,attn_pool],1))
        return torch.cat([hd(z) for hd in self.heads],dim=1)

_tidx = {t:i for i,t in enumerate(TARGETS)}
_MT_MU=np.array([train.loc[train.target_type==t,"target"].mean() for t in TARGETS],np.float32)
_MT_SD=np.array([train.loc[train.target_type==t,"target"].std()+1e-6 for t in TARGETS],np.float32)

def _mt_predict(model,graphs,bs=256):
    model.eval(); out=np.zeros((len(graphs),len(TARGETS)),np.float32)
    with torch.no_grad():
        for i2 in range(0,len(graphs),bs):
            X,EI,EA,B=collate(graphs[i2:i2+bs])
            X,EI,EA,B=X.to(DEVICE),EI.to(DEVICE),EA.to(DEVICE),B.to(DEVICE)
            out[i2:i2+bs]=model(X,EI,EA,B).cpu().numpy()
    return out

def train_multitask(seeds=(SEED,SEED+1,SEED+2), epochs=100, bs=64, lr=5e-4, patience=25):
    groups_all=train["smiles_canon"].values; tt=train["target_type"].values
    y=train["target"].values.astype(np.float32)
    ymat=np.full((len(train),len(TARGETS)),np.nan,np.float32)
    for i2 in range(len(train)): ymat[i2,_tidx[tt[i2]]]=y[i2]
    ynorm=(ymat-_MT_MU)/_MT_SD
    row_of={t:np.where(tt==t)[0] for t in TARGETS}
    pos_in={t:{r:i for i,r in enumerate(row_of[t])} for t in TARGETS}
    oof={t:np.full(len(row_of[t]),np.nan,np.float32) for t in TARGETS}
    tst={t:np.zeros(len(test),np.float32) for t in TARGETS}
    for seed in seeds:
        torch.manual_seed(seed); np.random.seed(seed)
        for fi_,(tr,va) in enumerate(make_group_folds(groups_all,N_FOLDS,seed=seed)):
            model=MultiTaskMPNN(ADIM,BDIM).to(DEVICE)
            opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-5)
            sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
            gtr=[G_tr[i] for i in tr]; Ytr=ynorm[tr]
            Mtr=torch.tensor(~np.isnan(Ytr)); Ytr0=torch.tensor(np.nan_to_num(Ytr))
            gva=[G_tr[i] for i in va]; idx=np.arange(len(tr)); best=-1e9; best_state=None; wait=0
            for ep in range(epochs):
                model.train(); np.random.shuffle(idx)
                for k in range(0,len(idx),bs):
                    bi=idx[k:k+bs]
                    if len(bi)<2: continue
                    X,EI,EA,B=collate([gtr[j] for j in bi])
                    X,EI,EA,B=X.to(DEVICE),EI.to(DEVICE),EA.to(DEVICE),B.to(DEVICE)
                    pred=model(X,EI,EA,B); mmask=Mtr[bi].to(DEVICE); tgt=Ytr0[bi].to(DEVICE)
                    diff=(pred-tgt)[mmask]
                    loss=F.smooth_l1_loss(diff,torch.zeros_like(diff))
                    opt.zero_grad(); loss.backward(); opt.step()
                sched.step()
                vp=_mt_predict(model,gva)*_MT_SD+_MT_MU; r2s=[]
                va_list=list(va)
                for j,t in enumerate(TARGETS):
                    rows=[r for r in va if tt[r]==t]
                    if len(rows)>=5:
                        yr=y[rows]; pr=vp[[va_list.index(r) for r in rows],j]; r2s.append(r2_score(yr,pr))
                r2=float(np.mean(r2s)) if r2s else -1e9
                if r2>best: best=r2; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
                else:
                    wait+=1
                    if wait>=patience: break
            model.load_state_dict(best_state)
            vp=_mt_predict(model,gva)*_MT_SD+_MT_MU
            for li,r in enumerate(va):
                t=tt[r]; j=_tidx[t]; p=pos_in[t][r]
                oof[t][p]=(0.0 if np.isnan(oof[t][p]) else oof[t][p])+vp[li,j]/len(seeds)
            MTL_ENCODER_STATE={k:v.detach().cpu().clone() for k,v in model.state_dict().items() if k.split('.')[0] in ('lin0','edge','msg','upd','bn','attn_w','trunk')}
            tp=_mt_predict(model,G_te)*_MT_SD+_MT_MU
            for j,t in enumerate(TARGETS): tst[t]+=tp[:,j]/(len(seeds)*N_FOLDS)
    res={}
    for t in TARGETS:
        yt=y[row_of[t]]; v=~np.isnan(oof[t])
        print(f"[{t}] multitask OOF R2 = {r2_score(yt[v],oof[t][v]):.4f}")
        res[t]={"oof":oof[t],"test":tst[t],"y":yt}
    return res

print("Training Multi-Task GNN ...")
MTL_ENCODER_STATE = None
mtl_res = train_multitask()
if MTL_ENCODER_STATE is not None:
    print(f"Captured multi-task encoder ({len(MTL_ENCODER_STATE)} tensors) for per-target warm-start")

Training Multi-Task GNN ...
[eea] multitask OOF R2 = 0.9250
[egb] multitask OOF R2 = 0.9465
[egc] multitask OOF R2 = 0.9220
[ei] multitask OOF R2 = 0.8612
[eps] multitask OOF R2 = 0.8402
[nc] multitask OOF R2 = 0.8904
[tg] multitask OOF R2 = 0.9099


## 5a. Dedicated per-target GNN heads — RESTORED

**This member existed in the earlier `papbol`/`dont-judge` builds and was dropped in the last
refactor — the single largest regression in this notebook.** Previously the per-target GNN was
the *best individual member* on egc (0.9300) and strong on tg (0.9160). In the current build the
graph family is represented only by the shared-trunk multi-task model (egc 0.9034, tg 0.8899),
and the best members are lgb_et (egc 0.9120, tg 0.9193).

The multi-task trunk optimises the average of seven losses, so its representation is a
compromise; a dedicated head specialises for one property and — the point for stacking — has
errors decorrelated from both the trees and the shared model. Each head warm-starts from the
multi-task encoder (52 shape-matched tensors), then specialises; small targets freeze the first
two message-passing layers and train at lr 2e-4 so ~220 rows cannot overwrite the inherited
representation.

In [12]:
"""Dedicated per-target GNN heads — RESTORED.

This member existed in the earlier `papbol` / `dont-judge` builds and was dropped in the last
refactor. Its absence is the single largest regression in this notebook: in the earlier run the
per-target GNN was the *best individual member* on egc (0.9300) and strong on tg (0.9160),
whereas the current build's best members are lgb_et 0.9120 (egc) and lgb_et 0.9193 (tg), with
only the multi-task GNN (`mtl`, egc 0.9034 / tg 0.8899) representing the graph family.

Why a per-target head adds something the multi-task head does not: the multi-task trunk is
optimised for the average of seven losses, so its representation is a compromise. A dedicated
head can specialise its pooling and trunk for one property, and — importantly for stacking —
its errors are decorrelated from both the trees and the shared-trunk model.

Warm-start: each per-target model is initialised from the multi-task encoder (shape-matched
layers only), so it inherits the representation learned across all seven properties, then
specialises. Small targets additionally freeze the first two message-passing layers and train
at a lower learning rate, since ~220 rows can otherwise overwrite the inherited encoder within
a couple of epochs.
"""

GNN_PER_TARGET_ENABLE = True
# egc/tg gain most (largest N, graph member was previously their best); ei/eps benefit from a
# specialised head on top of the shared representation.
GNN_TARGETS = [t for t in TARGETS if t in set(BIG_TARGETS) | {"ei", "eps", "nc", "eea", "egb"}]


class SingleTaskMPNN(nn.Module):
    """Same backbone as MultiTaskMPNN (so encoder weights transfer directly), one head."""
    def __init__(self, adim, bdim, hid=192, layers=4, drop=0.1):
        super().__init__(); self.L = layers
        self.lin0 = nn.Linear(adim, hid); self.edge = nn.Linear(bdim, hid)
        self.msg = nn.ModuleList([nn.Linear(2 * hid, hid) for _ in range(layers)])
        self.upd = nn.ModuleList([nn.GRUCell(hid, hid) for _ in range(layers)])
        self.bn = nn.ModuleList([nn.BatchNorm1d(hid) for _ in range(layers)])
        self.attn_w = nn.Linear(hid, 1)
        self.trunk = nn.Sequential(nn.Linear(3 * hid, hid), nn.ReLU(), nn.Dropout(drop))
        self.head = nn.Sequential(nn.Linear(hid, hid // 2), nn.ReLU(), nn.Linear(hid // 2, 1))

    def forward(self, X, EI, EA, B):
        h = F.relu(self.lin0(X)); e = self.edge(EA); s, d = EI[0], EI[1]
        for l in range(self.L):
            msg = torch.relu(self.msg[l](torch.cat([h[s], e], 1)))
            agg = torch.zeros_like(h).index_add_(0, d, msg)
            h = self.bn[l](self.upd[l](agg, h))
        ng = int(B.max().item()) + 1
        summ = torch.zeros(ng, h.size(1), device=h.device).index_add_(0, B, h)
        cnt = torch.zeros(ng, 1, device=h.device).index_add_(0, B,
                    torch.ones(h.size(0), 1, device=h.device))
        mean = summ / cnt.clamp(min=1)
        mx = torch.full((ng, h.size(1)), -1e9, device=h.device).index_reduce_(
                    0, B, h, "amax", include_self=True)
        _sc = self.attn_w(h).squeeze(-1)
        _ex = torch.exp(_sc - _sc.max())
        _dn = torch.zeros(ng, device=h.device).index_add_(0, B, _ex).clamp(min=1e-8)
        _w = (_ex / _dn[B]).unsqueeze(-1)
        attn = torch.zeros(ng, h.size(1), device=h.device).index_add_(0, B, h * _w)
        return self.head(self.trunk(torch.cat([mean, mx, attn], 1))).squeeze(-1)


def _st_predict(model, graphs, bs=256):
    model.eval(); out = np.zeros(len(graphs), np.float32)
    with torch.no_grad():
        for i in range(0, len(graphs), bs):
            X, EI, EA, B = collate(graphs[i:i + bs])
            X, EI, EA, B = X.to(DEVICE), EI.to(DEVICE), EA.to(DEVICE), B.to(DEVICE)
            out[i:i + bs] = model(X, EI, EA, B).cpu().numpy()
    return out


def _warm_start_single(model):
    """Copy shape-matching encoder weights from the trained multi-task model."""
    src = globals().get("MTL_ENCODER_STATE")
    if src is None:
        return 0
    md = model.state_dict(); n = 0
    for k, v in src.items():
        if k in md and md[k].shape == v.shape:
            md[k] = v; n += 1
    model.load_state_dict(md)
    return n


def train_gnn_target(target, seeds=(SEED, SEED + 1), epochs=120, patience=25):
    is_small = target in SMALL_TARGETS
    lr = 2e-4 if is_small else 5e-4
    wd = 1e-4 if is_small else 1e-5
    bs = 32 if is_small else 64

    mask = (train["target_type"].values == target)
    rows = np.where(mask)[0]
    y = train.loc[mask, "target"].values.astype(np.float32)
    groups = train.loc[mask, "smiles_canon"].values
    gsub = [G_tr[r] for r in rows]

    oof = np.zeros(len(y)); tst = np.zeros(len(test)); nrun = 0
    for seed in seeds:
        torch.manual_seed(seed); np.random.seed(seed)
        for tr_i, va_i in make_group_folds(groups, N_FOLDS, seed=seed):
            mu, sd = y[tr_i].mean(), y[tr_i].std() + 1e-8
            ytr_n = (y[tr_i] - mu) / sd
            model = SingleTaskMPNN(ADIM, BDIM).to(DEVICE)
            _warm_start_single(model)
            if is_small:
                # protect the inherited representation from ~220 rows of gradient
                for p in model.lin0.parameters(): p.requires_grad = False
                for p in model.edge.parameters(): p.requires_grad = False
                for li in range(min(2, len(model.msg))):
                    for p in model.msg[li].parameters(): p.requires_grad = False
                    for p in model.upd[li].parameters(): p.requires_grad = False
                    for p in model.bn[li].parameters(): p.requires_grad = False
            opt = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                                    lr=lr, weight_decay=wd)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
            gtr = [gsub[i] for i in tr_i]; gva = [gsub[i] for i in va_i]
            idx = np.arange(len(gtr)); best = -1e9; best_state = None; wait = 0
            for ep in range(epochs):
                model.train(); np.random.shuffle(idx)
                for i in range(0, len(idx), bs):
                    bi = idx[i:i + bs]
                    if len(bi) < 2: continue
                    X, EI, EA, B = collate([gtr[j] for j in bi])
                    X, EI, EA, B = X.to(DEVICE), EI.to(DEVICE), EA.to(DEVICE), B.to(DEVICE)
                    loss = F.smooth_l1_loss(model(X, EI, EA, B),
                                            torch.tensor(ytr_n[bi], device=DEVICE))
                    opt.zero_grad(); loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
                sch.step()
                r2v = r2_score(y[va_i], _st_predict(model, gva) * sd + mu)
                if r2v > best:
                    best = r2v; wait = 0
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                else:
                    wait += 1
                    if wait >= patience: break
            if best_state is not None: model.load_state_dict(best_state)
            oof[va_i] += (_st_predict(model, gva) * sd + mu) / len(seeds)
            tst += (_st_predict(model, G_te) * sd + mu) / (len(seeds) * N_FOLDS)
            nrun += 1
    r2 = r2_score(y, oof)
    print(f"[{target}] per-target GNN OOF R2 = {r2:.4f}  ({nrun} fold-runs, "
          f"{'small: lr2e-4+frozen' if is_small else 'big: lr5e-4'})")
    return {"oof": oof, "test": tst}


gnn_res = {}
if GNN_PER_TARGET_ENABLE:
    print("=" * 70)
    print("Dedicated per-target GNN heads (restored)")
    print("=" * 70)
    for t in GNN_TARGETS:
        _t0 = time.time()
        gnn_res[t] = train_gnn_target(t)
        print(f"       [timing] {t}: {(time.time()-_t0)/60:.1f} min")
else:
    print("Per-target GNN disabled.")


Dedicated per-target GNN heads (restored)
[eea] per-target GNN OOF R2 = 0.9453  (16 fold-runs, small: lr2e-4+frozen)
       [timing] eea: 0.9 min
[egb] per-target GNN OOF R2 = 0.9418  (16 fold-runs, small: lr2e-4+frozen)
       [timing] egb: 1.4 min
[egc] per-target GNN OOF R2 = 0.9292  (16 fold-runs, big: lr5e-4)
       [timing] egc: 6.0 min
[ei] per-target GNN OOF R2 = 0.8600  (16 fold-runs, small: lr2e-4+frozen)
       [timing] ei: 1.1 min
[eps] per-target GNN OOF R2 = 0.8353  (16 fold-runs, small: lr2e-4+frozen)
       [timing] eps: 0.9 min
[nc] per-target GNN OOF R2 = 0.8869  (16 fold-runs, small: lr2e-4+frozen)
       [timing] nc: 1.0 min
[tg] per-target GNN OOF R2 = 0.9171  (16 fold-runs, big: lr5e-4)
       [timing] tg: 19.3 min


## 5b. Additional decorrelated stack members (all seven targets)

Previous run's pruning log showed the big targets were effectively stacking four correlated
GBMs plus one GNN — `ridge` was pruned every time (tg: 0.8346 vs 0.9194 best). These three
additions give **every** target at least one kernel-based and one randomized-tree member:

* **Multi-kernel GPR** (small targets) — GPR already beat the tree blend on `eps` last run
  (0.8423 vs 0.8199). Rather than committing to one Matérn, average Matérn-1.5, Matérn-2.5 and
  RationalQuadratic; kernel choice dominates GPR behaviour and averaging beats selecting it on
  229 rows.
* **Nyström + KernelRidge** (all targets) — exact GPR is O(n³) and cannot run on tg's 4139
  rows; the Nyström approximation makes an RBF model affordable there, so the big targets get
  a kernel member for the first time. γ from the median-pairwise-distance heuristic per fold.
* **ExtraTrees** (all targets) — fully randomized split thresholds, a different bias from the
  boosted members, no early stopping needed.

In [13]:
"""New decorrelated stack members, added for EVERY target (not just ei/eps).

Motivation from the previous run's per-target logs: on the big targets (tg/egc) the only
non-tree member surviving the 0.10 pruning gate was `mtl` — ridge got dropped every time
(tg 0.8346 vs 0.9194 best). So tg/egc were effectively stacking 4 correlated GBMs + 1 GNN.
Adding a *kernel* member and an *extra-randomized-tree* member gives the meta-learner two
genuinely different error structures on the large targets too.

Three additions:
  1. Multi-kernel GPR (small targets) — the previous run showed GPR already beating the tree
     blend on eps (0.8423 vs 0.8199). Instead of one Matern(1.5), average three kernels with
     different smoothness priors; kernel choice is the dominant hyperparameter for GPR and
     averaging over it is cheaper and safer than selecting on 229 rows.
  2. Nystroem + KernelRidge (ALL targets) — exact GPR is O(n^3) so it cannot run on tg's 4139
     rows. Nystroem random-feature approximation makes an RBF kernel model affordable at that
     size, giving the big targets their first real kernel member.
  3. ExtraTrees (ALL targets) — fully randomized split thresholds, a different bias from the
     gradient-boosted members, and it needs no early stopping.
"""
from sklearn.gaussian_process.kernels import RationalQuadratic, ConstantKernel
from sklearn.kernel_approximation import Nystroem
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import make_pipeline


def _lean_idx(target, n_top_var=30):
    """Low-dimensional feature view for kernel models: physics/cross/backbone columns (always
    informative, few dims) + the highest-variance dense descriptors. Wide fingerprint matrices
    make distance-based models collapse, so this stays ~40-70 dims."""
    mask = (train["target_type"].values == target)
    dense_idx_full = [i for i, c in enumerate(ALL_COLS) if c in DENSE_COLS]
    force_idx = [i for i, c in enumerate(ALL_COLS) if str(c).startswith(("phys_", "cross_", "x_"))]
    var = Xtr_all[mask][:, dense_idx_full].var(0)
    top_var = np.array(dense_idx_full)[np.argsort(-var)[:n_top_var]].tolist()
    return sorted(set(force_idx + top_var))


# ------------------------------------------------------------------ 1. multi-kernel GPR
def _gpr_kernels():
    return [
        ("m15", ConstantKernel(1.0) * Matern(length_scale=1.0, nu=1.5) + WhiteKernel(noise_level=1e-3)),
        ("m25", ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(noise_level=1e-3)),
        ("rq",  ConstantKernel(1.0) * RationalQuadratic(length_scale=1.0, alpha=1.0) + WhiteKernel(noise_level=1e-3)),
    ]


def train_gpr_multi(target):
    """Average predictions over three kernel priors instead of committing to one."""
    mask = (train["target_type"].values == target)
    idx = _lean_idx(target)
    X = Xtr_all[mask][:, idx]
    y = train.loc[mask, "target"].values.astype(np.float32)
    Xt = Xte_all[:, idx]
    groups = train.loc[mask, "smiles_canon"].values
    kerns = _gpr_kernels()
    oof = np.zeros(len(y)); tst = np.zeros(len(test)); used = 0
    for kname, kern in kerns:
        k_oof = np.zeros(len(y)); k_tst = np.zeros(len(test)); ok = True
        for tr, va in make_group_folds(groups, N_FOLDS, seed=SEED):
            try:
                sc = StandardScaler().fit(X[tr])
                mdl = GaussianProcessRegressor(kernel=kern, alpha=0.05,
                                               random_state=SEED, normalize_y=True)
                mdl.fit(sc.transform(X[tr]), y[tr])
                k_oof[va] = mdl.predict(sc.transform(X[va]))
                k_tst += mdl.predict(sc.transform(Xt)) / N_FOLDS
            except Exception as e:
                print(f"  [{target}] GPR-{kname} fold failed ({type(e).__name__}); kernel skipped")
                ok = False
                break
        if ok:
            print(f"  [{target}] gpr-{kname} OOF R2 = {r2_score(y, k_oof):.4f}")
            oof += k_oof; tst += k_tst; used += 1
    if used == 0:
        return None
    oof /= used; tst /= used
    print(f"[{target}] gpr-multi ({used} kernels) OOF R2 = {r2_score(y, oof):.4f}")
    return {"oof": oof, "test": tst}


# ------------------------------------------------- 2. Nystroem-approximated kernel ridge
def train_nystroem_krr(target):
    """RBF kernel ridge via Nystroem features — affordable on the big targets where exact
    GPR is not. gamma is set from the median pairwise-distance heuristic per fold."""
    mask = (train["target_type"].values == target)
    idx = _lean_idx(target, n_top_var=60)
    X = Xtr_all[mask][:, idx]
    y = train.loc[mask, "target"].values.astype(np.float32)
    Xt = Xte_all[:, idx]
    groups = train.loc[mask, "smiles_canon"].values
    n_comp = int(min(300, max(50, 0.5 * len(y))))
    oof = np.zeros(len(y)); tst = np.zeros(len(test))
    for tr, va in make_group_folds(groups, N_FOLDS, seed=SEED):
        try:
            sc = StandardScaler().fit(X[tr])
            Xtr_s, Xva_s, Xte_s = sc.transform(X[tr]), sc.transform(X[va]), sc.transform(Xt)
            # median-distance heuristic for gamma (on a subsample, for speed)
            sub = Xtr_s[np.random.RandomState(SEED).choice(
                len(Xtr_s), size=min(300, len(Xtr_s)), replace=False)]
            d2 = ((sub[:, None, :] - sub[None, :, :]) ** 2).sum(-1)
            med = np.median(d2[d2 > 0]) if (d2 > 0).any() else 1.0
            gamma = 1.0 / max(med, 1e-6)
            ymu, ysd = y[tr].mean(), y[tr].std() + 1e-8
            pipe = make_pipeline(
                Nystroem(gamma=gamma, n_components=min(n_comp, len(Xtr_s)), random_state=SEED),
                KernelRidge(alpha=1.0, kernel="linear"),
            )
            pipe.fit(Xtr_s, (y[tr] - ymu) / ysd)
            oof[va] = pipe.predict(Xva_s) * ysd + ymu
            tst += (pipe.predict(Xte_s) * ysd + ymu) / N_FOLDS
        except Exception as e:
            print(f"  [{target}] nyskrr fold failed ({type(e).__name__}); filling with mean")
            oof[va] = y[tr].mean()
            tst += y[tr].mean() / N_FOLDS
    print(f"[{target}] nyskrr OOF R2 = {r2_score(y, oof):.4f}")
    return {"oof": oof, "test": tst}


# ------------------------------------------------------------------------ 3. ExtraTrees
def train_extratrees(target):
    """Fully randomized split thresholds -> different bias from the boosted members.
    Uses the same per-target selected feature set as the trees."""
    mask = (train["target_type"].values == target)
    feat_idx = SELECTED[target]
    X = Xtr_all[mask][:, feat_idx]
    y = train.loc[mask, "target"].values.astype(np.float32)
    Xt = Xte_all[:, feat_idx]
    groups = train.loc[mask, "smiles_canon"].values
    is_small = target in SMALL_TARGETS
    oof = np.zeros(len(y)); tst = np.zeros(len(test))
    for tr, va in make_group_folds(groups, N_FOLDS, seed=SEED):
        mdl = ExtraTreesRegressor(
            n_estimators=600,
            max_features=0.3,
            min_samples_leaf=(3 if is_small else 1),
            max_depth=(12 if is_small else None),
            random_state=SEED, n_jobs=-1)
        mdl.fit(X[tr], y[tr])
        oof[va] = mdl.predict(X[va])
        tst += mdl.predict(Xt) / N_FOLDS
    print(f"[{target}] extratrees OOF R2 = {r2_score(y, oof):.4f}")
    return {"oof": oof, "test": tst}


print("=" * 70)
print("Multi-kernel GPR (small targets)")
print("=" * 70)
gpr_res = {}
for t in SMALL_TARGETS:
    _r = train_gpr_multi(t)
    if _r is not None:
        gpr_res[t] = _r

print("\n" + "=" * 70)
print("Nystroem-KRR (all targets)")
print("=" * 70)
nyskrr_res = {}
for t in TARGETS:
    _t0 = time.time()
    nyskrr_res[t] = train_nystroem_krr(t)
    print(f"       [timing] {t}: {(time.time()-_t0)/60:.1f} min")

print("\n" + "=" * 70)
print("ExtraTrees (all targets)")
print("=" * 70)
et_res = {}
for t in TARGETS:
    _t0 = time.time()
    et_res[t] = train_extratrees(t)
    print(f"       [timing] {t}: {(time.time()-_t0)/60:.1f} min")


Multi-kernel GPR (small targets)
  [eea] gpr-m15 OOF R2 = 0.9081
  [eea] gpr-m25 OOF R2 = 0.9071
  [eea] gpr-rq OOF R2 = 0.9058
[eea] gpr-multi (3 kernels) OOF R2 = 0.9071
  [egb] gpr-m15 OOF R2 = 0.9309
  [egb] gpr-m25 OOF R2 = 0.9300
  [egb] gpr-rq OOF R2 = 0.9291
[egb] gpr-multi (3 kernels) OOF R2 = 0.9301
  [ei] gpr-m15 OOF R2 = 0.8480
  [ei] gpr-m25 OOF R2 = 0.8463
  [ei] gpr-rq OOF R2 = 0.8449
[ei] gpr-multi (3 kernels) OOF R2 = 0.8466
  [eps] gpr-m15 OOF R2 = 0.8423
  [eps] gpr-m25 OOF R2 = 0.8427
  [eps] gpr-rq OOF R2 = 0.8432
[eps] gpr-multi (3 kernels) OOF R2 = 0.8429
  [nc] gpr-m15 OOF R2 = 0.8884
  [nc] gpr-m25 OOF R2 = 0.8887
  [nc] gpr-rq OOF R2 = 0.8859
[nc] gpr-multi (3 kernels) OOF R2 = 0.8888

Nystroem-KRR (all targets)
[eea] nyskrr OOF R2 = 0.8362
       [timing] eea: 0.0 min
[egb] nyskrr OOF R2 = 0.8768
       [timing] egb: 0.0 min
[egc] nyskrr OOF R2 = 0.8309
       [timing] egc: 0.1 min
[ei] nyskrr OOF R2 = 0.8012
       [timing] ei: 0.0 min
[eps] nyskrr OOF R2 = 

## 5d. Graph transformer members (Graphormer-style)

Added without touching anything above. This notebook's existing members all pass
messages locally: the MPNN has a 4-hop receptive field. Measured on this dataset,
median graph diameter is **17**, median backbone length **15**, and **80.6% of
molecules have diameter > 8** — so on four molecules in five the MPNN never sees
one end of the repeat unit from the other. For band gaps, where conjugation
length along the backbone *is* the property, that is a structural blind spot.

The transformer attends over all atom pairs at layer 1, biased by **shortest-path
distance**, so it sees the whole molecule while still knowing how far apart atoms
are. Plus:

* **RWSE** — random-walk structural encoding, diag(P^k) for k=1..8, on GPU.
* **Polymer positional encoding** — each atom's normalised distance to each `*`
  endpoint and a backbone flag, so it can attend *along the chain*.

In a separate build this was the **best single member on egb (0.9326) and egc
(0.9151)** and earned 0.15–0.37 of the blend weight on five of seven targets.

It registers as `gt_res` / `gtmtl_res` in exactly the same shape as `gnn_res`,
so `stack()` picks it up and the existing meta-learner decides whether to use
it. **If this cell fails or is skipped, the notebook behaves exactly as before.**


In [14]:
# ══════════════════════════════════════════════════════════════════════
# 5d. GRAPH TRANSFORMER  — additive member, nothing above is modified
# ══════════════════════════════════════════════════════════════════════
GT_ENABLE     = True
GT_PI1M_N     = 200_000      # unlabelled molecules for pretraining (0 = skip)
GT_PRE_EPOCHS = 3
GT_HID, GT_L, GT_HEADS = 192, 4, 8
SPD_CAP, RWSE_K, GT_MAXN, POLY_DIM = 15, 8, 128, 3
GT_TARGETS = list(TARGETS)

import math, gc, time
from rdkit.Chem import Descriptors as _D

_EL = ["C","N","O","S","F","Cl","Br","I","Si","P","B","Se","*"]
_E2I = {e:i for i,e in enumerate(_EL)}
_HYB = [Chem.HybridizationType.SP, Chem.HybridizationType.SP2, Chem.HybridizationType.SP3,
        Chem.HybridizationType.SP3D, Chem.HybridizationType.SP3D2]
_H2I = {h:i for i,h in enumerate(_HYB)}
NEL, NHY = len(_EL)+1, len(_HYB)+1
GT_ADIM = NEL + 7 + 5 + 6 + NHY + 3


def mol_to_gt(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None or m.GetNumAtoms() < 2 or m.GetNumAtoms() > GT_MAXN: return None
    n = m.GetNumAtoms(); A = np.zeros((n,6), np.uint8)
    for a in m.GetAtoms():
        i = a.GetIdx()
        A[i,0] = _E2I.get(a.GetSymbol(), len(_EL))
        A[i,1] = min(a.GetDegree(), 6)
        A[i,2] = int(np.clip(a.GetFormalCharge(), -2, 2) + 2)
        A[i,3] = min(a.GetTotalNumHs(), 5)
        A[i,4] = _H2I.get(a.GetHybridization(), len(_HYB))
        A[i,5] = (int(a.GetIsAromatic()) | (int(a.IsInRing())<<1) | (int(a.GetAtomicNum()==0)<<2))
    try: D = Chem.GetDistanceMatrix(m)
    except Exception: return None
    D = np.where(np.isfinite(D) & (D < 1e6), D, SPD_CAP+1)
    spd = np.clip(D, 0, SPD_CAP+1).astype(np.uint8)
    BT = {Chem.BondType.SINGLE:1, Chem.BondType.DOUBLE:2,
          Chem.BondType.TRIPLE:3, Chem.BondType.AROMATIC:4}
    bt = np.zeros((n,n), np.uint8)
    for b in m.GetBonds():
        i,j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        c = BT.get(b.GetBondType(), 1); bt[i,j] = c; bt[j,i] = c
    adj = (spd == 1).astype(np.uint8)
    st = [a.GetIdx() for a in m.GetAtoms() if a.GetAtomicNum() == 0]
    poly = np.zeros((n, POLY_DIM), np.float32)
    if len(st) == 2:
        dh, dt = D[st[0]], D[st[1]]; L = max(float(D[st[0], st[1]]), 1.0)
        poly[:,0] = np.clip(dh/L, 0, 2); poly[:,1] = np.clip(dt/L, 0, 2)
        poly[:,2] = (np.abs(dh+dt-L) < 1e-6).astype(np.float32)
    return A, spd, bt, adj, poly


def gt_collate(items, device):
    B = len(items); N = max(g[0].shape[0] for g in items)
    A = np.zeros((B,N,6), np.uint8); spd = np.full((B,N,N), SPD_CAP+1, np.uint8)
    bt = np.zeros((B,N,N), np.uint8); adj = np.zeros((B,N,N), np.float32)
    poly = np.zeros((B,N,POLY_DIM), np.float32); mask = np.zeros((B,N), np.float32)
    for k,(a,s,b,ad,p) in enumerate(items):
        n = a.shape[0]
        A[k,:n]=a; spd[k,:n,:n]=s; bt[k,:n,:n]=b; adj[k,:n,:n]=ad; poly[k,:n]=p; mask[k,:n]=1.
    t = lambda x,d: torch.from_numpy(x).to(device=device, dtype=d)
    Ai = t(A, torch.long)
    x = torch.cat([F.one_hot(Ai[...,0], NEL), F.one_hot(Ai[...,1], 7),
                   F.one_hot(Ai[...,2], 5), F.one_hot(Ai[...,3], 6),
                   F.one_hot(Ai[...,4], NHY),
                   ((Ai[...,5:6] >> torch.arange(3, device=device)) & 1)], -1).float()
    return (x, t(spd, torch.long), t(bt, torch.long), t(adj, torch.float32),
            t(poly, torch.float32), t(mask, torch.float32), Ai[...,0])


def _rwse(adj, mask, k=RWSE_K):
    P = adj / adj.sum(-1, keepdim=True).clamp_min(1.0)
    out, M = [], P
    for _ in range(k):
        out.append(torch.diagonal(M, dim1=-2, dim2=-1)); M = M @ P
    return torch.stack(out, -1) * mask.unsqueeze(-1)


class _GTLayer(nn.Module):
    def __init__(self, h, heads, drop=0.1):
        super().__init__(); self.h, self.nh, self.dk = h, heads, h//heads
        self.qkv = nn.Linear(h, 3*h); self.proj = nn.Linear(h, h)
        self.n1, self.n2 = nn.LayerNorm(h), nn.LayerNorm(h)
        self.ff = nn.Sequential(nn.Linear(h, 2*h), nn.GELU(), nn.Dropout(drop), nn.Linear(2*h, h))
        self.drop = nn.Dropout(drop)
    def forward(self, x, bias, km):
        B, N, _ = x.shape
        q,k,v = self.qkv(self.n1(x)).chunk(3, -1)
        sh = lambda t: t.view(B,N,self.nh,self.dk).transpose(1,2)
        q,k,v = sh(q), sh(k), sh(v)
        att = (q @ k.transpose(-2,-1))/math.sqrt(self.dk) + bias
        att = att.masked_fill(km.view(B,1,1,N) == 0, -1e4)
        o = (self.drop(att.softmax(-1)) @ v).transpose(1,2).reshape(B,N,self.h)
        x = x + self.drop(self.proj(o))
        return x + self.drop(self.ff(self.n2(x)))


class GraphTransformer(nn.Module):
    def __init__(self, hid=GT_HID, layers=GT_L, heads=GT_HEADS, drop=0.1):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(GT_ADIM+RWSE_K+POLY_DIM, hid), nn.GELU(),
                                 nn.Linear(hid, hid))
        self.spd_bias = nn.Embedding(SPD_CAP+2, heads); self.bt_bias = nn.Embedding(5, heads)
        self.layers = nn.ModuleList([_GTLayer(hid, heads, drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(hid)
        self.out = nn.Sequential(nn.Linear(2*hid, hid), nn.GELU(), nn.Dropout(drop),
                                 nn.Linear(hid, hid))
        self.hid = hid
    def forward(self, x, spd, bt, adj, poly, mask, return_nodes=False):
        h = self.inp(torch.cat([x, _rwse(adj, mask), poly], -1)) * mask.unsqueeze(-1)
        bias = (self.spd_bias(spd) + self.bt_bias(bt)).permute(0,3,1,2)
        for L in self.layers: h = L(h, bias, mask)
        h = self.norm(h) * mask.unsqueeze(-1)
        c = mask.sum(1, keepdim=True).clamp_min(1)
        mean = h.sum(1)/c
        mx = h.masked_fill(mask.unsqueeze(-1) == 0, -1e4).max(1).values
        g = self.out(torch.cat([mean, mx], -1))
        return (g, h) if return_nodes else g


class GTHead(nn.Module):
    def __init__(self, ntask=1, hid=GT_HID, enc=None):
        super().__init__()
        self.enc = enc if enc is not None else GraphTransformer()
        self.heads = nn.ModuleList([nn.Sequential(nn.Linear(hid,128), nn.GELU(),
                                    nn.Dropout(0.1), nn.Linear(128,1)) for _ in range(ntask)])
    def forward(self, *a):
        g = self.enc(*a); return torch.cat([h(g) for h in self.heads], 1)


GTG_tr, GTG_te, GT_ENC = None, None, None
if GT_ENABLE:
    GTG_tr = [mol_to_gt(s) for s in train["smiles"]]
    GTG_te = [mol_to_gt(s) for s in test["smiles"]]
    print(f"graph-transformer encoding: train {sum(g is not None for g in GTG_tr)}/{len(GTG_tr)}"
          f" | test {sum(g is not None for g in GTG_te)}/{len(GTG_te)}")


graph-transformer encoding: train 7404/7405 | test 4939/4940


In [15]:
# ── PI1M pretraining of the transformer (skip cleanly if unavailable) ──
_SSL = ["MolWt","MolLogP","MolMR","TPSA","NumRotatableBonds","RingCount","NumAromaticRings",
        "FractionCSP3","NumHAcceptors","NumHDonors","HeavyAtomCount","NHOHCount","NOCount",
        "NumAliphaticRings","NumSaturatedRings","BalabanJ","BertzCT","Chi0v","Chi1v","Chi2v",
        "Kappa1","Kappa2","Kappa3","HallKierAlpha","LabuteASA","qed","NumHeteroatoms",
        "NumValenceElectrons"]
_SFN = {n:f for n,f in Descriptors.descList if n in set(_SSL)}
_SORD = [n for n in _SSL if n in _SFN]

def _ssl_row(s):
    g = mol_to_gt(s)
    if g is None: return None
    m = Chem.MolFromSmiles(str(s))
    if m is None: return None
    r = []
    for n in _SORD:
        try:
            v = float(_SFN[n](m)); r.append(v if np.isfinite(v) else 0.0)
        except Exception: r.append(0.0)
    return g, np.asarray(r, np.float32)

if GT_ENABLE and GT_PI1M_N > 0 and os.path.exists(PI1M_PATH):
    try:
        _t0 = time.time()
        _pdf = pd.read_csv(PI1M_PATH)
        _c = "SMILES" if "SMILES" in _pdf.columns else _pdf.columns[0]
        _sm = _pdf[_c].astype(str).drop_duplicates().values; del _pdf; gc.collect()
        _sel = _sm[np.random.RandomState(SEED+3).permutation(len(_sm))[:GT_PI1M_N]]
        _rows = [r for r in (_ssl_row(s) for s in _sel) if r is not None]
        if len(_rows) < 5000:
            print("  too few PI1M molecules -> transformer trains from scratch")
        else:
            G = [r[0] for r in _rows]; Y = np.vstack([r[1] for r in _rows])
            _med = np.median(Y,0); _q1,_q3 = np.percentile(Y,[25,75],axis=0)
            Y = np.clip((Y-_med)/np.clip(_q3-_q1,1e-6,None), -8, 8).astype(np.float32)
            print(f"  PI1M: {len(G):,} graphs, SSL block {Y.shape} "
                  f"(prep {(time.time()-_t0)/60:.1f} min)")
            enc = GraphTransformer().to(DEVICE)
            hd = nn.Linear(GT_HID, Y.shape[1]).to(DEVICE)
            mh = nn.Linear(GT_HID, NEL).to(DEVICE)
            opt = torch.optim.AdamW(list(enc.parameters())+list(hd.parameters())
                                    + list(mh.parameters()), lr=8e-4, weight_decay=1e-2)
            Yt = torch.from_numpy(Y); BS = 128
            def _stream(n, bs, seed):
                r = np.random.RandomState(seed)
                while True:
                    o = r.permutation(n)
                    for i in range(0, n-bs+1, bs): yield o[i:i+bs]
            _gen = _stream(len(G), BS, SEED)
            def _step(idx):
                x,spd,bt,adj,poly,mask,elem = gt_collate([G[j] for j in idx], DEVICE)
                yb = Yt[idx].to(DEVICE)
                mk = (torch.rand_like(mask) < 0.15)*mask
                xm = x.clone(); xm[...,:NEL] = xm[...,:NEL]*(1-mk).unsqueeze(-1)
                g,h = enc(xm, spd, bt, adj, poly, mask, return_nodes=True)
                loss = F.smooth_l1_loss(hd(g), yb)
                if mk.bool().any(): loss = loss + 0.3*F.cross_entropy(mh(h[mk.bool()]), elem[mk.bool()])
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(enc.parameters(), 5.0); opt.step()
                return float(loss)
            for pg in opt.param_groups: pg["lr"] = 1.5e-4
            _t1 = time.time()
            for _ in range(20): _step(next(_gen))
            _st = (time.time()-_t1)/20
            _steps = max(1, len(G)//BS)*GT_PRE_EPOCHS
            print(f"  {_st*1000:.0f} ms/step -> {_steps} steps "
                  f"(~{_steps*_st/60:.0f} min)")
            _sch = torch.optim.lr_scheduler.OneCycleLR(opt, 8e-4, total_steps=_steps, pct_start=0.1)
            _run = 0.0
            for _i in range(_steps):
                _run += _step(next(_gen)); _sch.step()
                if (_i+1) % 500 == 0: print(f"    step {_i+1}/{_steps} loss {_run/(_i+1):.4f}", flush=True)
            GT_ENC = enc.eval()
            print(f"  pretraining done, loss {_run/_steps:.4f} "
                  f"({(time.time()-_t0)/60:.1f} min total)")
            del G, Y, Yt, _rows; gc.collect(); torch.cuda.empty_cache()
    except Exception as _e:
        import traceback; print("  PI1M pretraining failed, continuing:", _e); traceback.print_exc(limit=3)
print("graph-transformer encoder pretrained:", GT_ENC is not None)


  PI1M: 198,574 graphs, SSL block (198574, 28) (prep 21.9 min)
  81 ms/step -> 4653 steps (~6 min)
    step 500/4653 loss 0.1710
    step 1000/4653 loss 0.1050
    step 1500/4653 loss 0.0781
    step 2000/4653 loss 0.0636
    step 2500/4653 loss 0.0544
    step 3000/4653 loss 0.0480
    step 3500/4653 loss 0.0433
    step 4000/4653 loss 0.0396
    step 4500/4653 loss 0.0367
  pretraining done, loss 0.0359 (27.1 min total)
graph-transformer encoder pretrained: True


In [16]:
# ── per-target transformer heads, same fold protocol as the other members ──
def _gt_predict(model, graphs, bs=128, ntask=1):
    model.eval(); out = np.zeros((len(graphs), ntask), np.float32)
    with torch.no_grad():
        for i in range(0, len(graphs), bs):
            ch = graphs[i:i+bs]
            ok = [j for j,g in enumerate(ch) if g is not None]
            if not ok: continue
            a = gt_collate([ch[j] for j in ok], DEVICE)
            q = model(*a[:6]).float().cpu().numpy()
            for r,j in enumerate(ok): out[i+j] = q[r]
    return out


def train_gt_target(target, seeds=(SEED,), epochs=110, patience=18):
    is_small = target in SMALL_TARGETS
    lr_e = 1.5e-4 if is_small else 3e-4
    bs = 32 if is_small else 96
    mask = (train["target_type"].values == target)
    rows = np.where(mask)[0]
    y = train.loc[mask, "target"].values.astype(np.float32)
    groups = train.loc[mask, "smiles_canon"].values
    gsub = [GTG_tr[r] for r in rows]
    ok_rows = [i for i,g in enumerate(gsub) if g is not None]
    if len(ok_rows) < 0.9*len(gsub):
        print(f"[{target}] only {len(ok_rows)}/{len(gsub)} encodable -> skipping gt")
        return None
    oof = np.zeros(len(y)); tst = np.zeros(len(test)); nrun = 0
    for seed in seeds:
        torch.manual_seed(seed); np.random.seed(seed)
        for tr_i, va_i in make_group_folds(groups, N_FOLDS, seed=seed):
            tr_i = np.array([i for i in tr_i if gsub[i] is not None])
            va_i = np.array([i for i in va_i if gsub[i] is not None])
            if len(tr_i) < 20 or len(va_i) < 2: continue
            mu, sd = y[tr_i].mean(), y[tr_i].std()+1e-8
            ytr = (y[tr_i]-mu)/sd
            enc = GraphTransformer().to(DEVICE)
            if GT_ENC is not None: enc.load_state_dict(GT_ENC.state_dict())
            model = GTHead(1, enc=enc).to(DEVICE)
            opt = torch.optim.AdamW([{"params": model.enc.parameters(), "lr": lr_e},
                                     {"params": model.heads.parameters(), "lr": 1.2e-3}],
                                    weight_decay=1e-2)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
            gtr = [gsub[i] for i in tr_i]; gva = [gsub[i] for i in va_i]
            idx = np.arange(len(gtr)); best = -1e9; bstate = None; wait = 0
            for ep in range(epochs):
                model.train(); np.random.shuffle(idx)
                for i in range(0, len(idx), bs):
                    bi = idx[i:i+bs]
                    if len(bi) < 2: continue
                    a = gt_collate([gtr[j] for j in bi], DEVICE)
                    loss = F.smooth_l1_loss(model(*a[:6]).squeeze(1),
                                            torch.tensor(ytr[bi], device=DEVICE))
                    opt.zero_grad(); loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
                sch.step()
                if ep >= 15 and ep % 3 == 0:
                    r2v = r2_score(y[va_i], _gt_predict(model, gva).ravel()*sd + mu)
                    if r2v > best:
                        best = r2v; wait = 0
                        bstate = {k: v.cpu().clone() for k,v in model.state_dict().items()}
                    else:
                        wait += 1
                        if wait >= patience: break
            if bstate is not None: model.load_state_dict(bstate)
            oof[va_i] += (_gt_predict(model, gva).ravel()*sd + mu)/len(seeds)
            tst += (_gt_predict(model, GTG_te).ravel()*sd + mu)/(len(seeds)*N_FOLDS)
            nrun += 1
            del model, enc; gc.collect(); torch.cuda.empty_cache()
    if nrun == 0: return None
    print(f"[{target}] graph-transformer OOF R2 = {r2_score(y, oof):.4f}  ({nrun} fold-runs)")
    return {"oof": oof, "test": tst}


gt_res = {}
if GT_ENABLE and GTG_tr is not None:
    print("="*70); print("Graph transformer per-target heads"); print("="*70)
    for t in GT_TARGETS:
        _t0 = time.time()
        try:
            r = train_gt_target(t)
            if r is not None: gt_res[t] = r
        except Exception as _e:
            import traceback; print(f"[{t}] gt failed: {_e}"); traceback.print_exc(limit=3)
        print(f"       [timing] {t}: {(time.time()-_t0)/60:.1f} min", flush=True)
else:
    print("Graph transformer disabled.")


Graph transformer per-target heads
[eea] graph-transformer OOF R2 = 0.9328  (8 fold-runs)
       [timing] eea: 1.8 min
[egb] graph-transformer OOF R2 = 0.9241  (8 fold-runs)
       [timing] egb: 2.9 min
[egc] graph-transformer OOF R2 = 0.9159  (8 fold-runs)
       [timing] egc: 10.4 min
[ei] graph-transformer OOF R2 = 0.8003  (8 fold-runs)
       [timing] ei: 2.0 min
[eps] graph-transformer OOF R2 = 0.8146  (8 fold-runs)
       [timing] eps: 1.8 min
[nc] graph-transformer OOF R2 = 0.8788  (8 fold-runs)
       [timing] nc: 2.0 min
[tg] graph-transformer OOF R2 = 0.9034  (8 fold-runs)
       [timing] tg: 45.4 min


## 5e. GNN + CapsNet Hybrid — Routing-by-Agreement Readout

Every graph member so far reads a molecule out of its atoms with some flavour of weighted
averaging: mean pool, max pool, or a softmax **attention** score per atom (`mtl`), or attention
biased by shortest-path distance (`gt`). All of these boil an atom down to **one scalar** — "how
much this atom counts" — before summing. That throws away *which aspect* of the atom mattered.

**Capsule routing** (Sabour/Hinton dynamic routing, adapted here to atoms instead of pixels)
replaces the scalar score with a vote. Each atom's GNN hidden state is squashed into a small
*primary capsule* vector, then transformed by a separate weight matrix per output capsule to cast
a **vote** for what that output capsule's pose should be. Routing-by-agreement (a few iterations
of softmax + squash) lets an output capsule accept only the votes that agree with each other —
so instead of one attention weight, the readout is a small set of molecule-level capsules, each
implicitly specializing on a different structural motif, that settle by consensus rather than by
a single learned score.

This is added as a **new, independent stack member** (`capsnet`), not a replacement for anything:

- **Encoder**: identical message-passing shapes to `MultiTaskMPNN` (`lin0`/`edge`/`msg`/`upd`/`bn`),
  so each per-target head warm-starts from `MTL_ENCODER_STATE` exactly like the restored GNN heads
  in §5a — a message-passing trunk is not learnable from ~220 rows on the small targets.
- **Readout**: atoms → primary capsules → 3 rounds of dynamic routing → a fixed number of
  molecule-level capsules → small MLP head.
- Small targets additionally freeze the first two message-passing layers, same rationale as §5a.

The goal isn't for capsule routing to beat attention pooling outright — it's a genuinely different
inductive bias (voting/agreement vs. a softmax score), so its errors should decorrelate from `mtl`,
the per-target GNN heads, and the graph transformer, which is exactly what the meta-learner in
§6 is set up to exploit (it already prunes anything within 0.10 R² of the best member, so a weak
but *different* member costs nothing if it doesn't help).

Purely additive: registers as `capsnet_res[target] = {"oof":..., "test":...}` in the same shape as
every other member. `stack()` picks it up with one `if target in capsnet_res:` block. If this cell
is skipped or fails, nothing else in the notebook changes.


In [ ]:
"""GNN + CapsNet hybrid member — see markdown above for the rationale.

Architecture in one line: same message-passing trunk as MultiTaskMPNN, but atoms are read out
through capsule dynamic routing (agreement/voting) instead of mean/max/attention pooling.
"""

GNNCAPS_ENABLE = True
CAPS_DIM      = 16     # dim of each capsule's pose vector
NUM_CAPS      = 10     # number of molecule-level output capsules
ROUTING_ITERS = 3
CAPS_TARGETS  = list(TARGETS)   # every target gets a shot; train_capsnet_target skips low-coverage ones


def squash(s, dim=-1, eps=1e-8):
    sq = (s * s).sum(dim=dim, keepdim=True)
    return (sq / (1.0 + sq)) * s / torch.sqrt(sq + eps)


def to_dense(h, B, ng):
    """Un-flatten the (total_atoms, hid) tensor from `collate` into a padded (ng, maxN, hid)
    tensor + validity mask. Relies on `collate` laying atoms out in contiguous per-graph blocks
    in graph order, so a plain split + pad reconstructs per-graph atom sets with no gather/scatter."""
    counts = torch.bincount(B, minlength=ng).tolist()
    parts = torch.split(h, counts, dim=0)
    dense = nn.utils.rnn.pad_sequence(parts, batch_first=True)
    mask = nn.utils.rnn.pad_sequence(
        [torch.ones(c, device=h.device) for c in counts], batch_first=True)
    return dense, mask


class CapsuleReadout(nn.Module):
    """Atoms -> primary capsules -> routed-by-agreement molecule-level capsules."""
    def __init__(self, hid, caps_dim=CAPS_DIM, num_out=NUM_CAPS, iters=ROUTING_ITERS):
        super().__init__()
        self.in_proj = nn.Linear(hid, caps_dim)
        self.num_out, self.caps_dim, self.iters = num_out, caps_dim, iters
        self.W = nn.Parameter(0.01 * torch.randn(num_out, caps_dim, caps_dim))

    def forward(self, dense_h, mask):
        u = squash(self.in_proj(dense_h))                    # (B,N,Din) primary capsules
        u_hat = torch.einsum('bnd,odf->bnof', u, self.W)      # (B,N,O,Dout) votes
        Bsz, N, O, Dout = u_hat.shape
        b = torch.zeros(Bsz, N, O, device=dense_h.device)
        maskf = mask.unsqueeze(-1)
        v = None
        for it in range(self.iters):
            c = torch.softmax(b.masked_fill(maskf == 0, -1e9), dim=2) * maskf
            s = (c.unsqueeze(-1) * u_hat).sum(dim=1)          # (B,O,Dout)
            v = squash(s, dim=-1)
            if it < self.iters - 1:
                b = b + torch.einsum('bnof,bof->bno', u_hat, v)
        return v                                              # (B,O,Dout)


class GNNCapsNet(nn.Module):
    """Message-passing trunk shape-matched to MultiTaskMPNN (for warm-start) + capsule readout."""
    def __init__(self, adim, bdim, hid=192, layers=4, drop=0.1):
        super().__init__(); self.L = layers
        self.lin0 = nn.Linear(adim, hid); self.edge = nn.Linear(bdim, hid)
        self.msg = nn.ModuleList([nn.Linear(2 * hid, hid) for _ in range(layers)])
        self.upd = nn.ModuleList([nn.GRUCell(hid, hid) for _ in range(layers)])
        self.bn  = nn.ModuleList([nn.BatchNorm1d(hid) for _ in range(layers)])
        self.caps = CapsuleReadout(hid)
        self.head = nn.Sequential(nn.Linear(NUM_CAPS * CAPS_DIM, hid // 2), nn.ReLU(),
                                   nn.Dropout(drop), nn.Linear(hid // 2, 1))

    def forward(self, X, EI, EA, B):
        h = F.relu(self.lin0(X)); e = self.edge(EA); s, d = EI[0], EI[1]
        for l in range(self.L):
            msg = torch.relu(self.msg[l](torch.cat([h[s], e], 1)))
            agg = torch.zeros_like(h).index_add_(0, d, msg)
            h = self.bn[l](self.upd[l](agg, h))
        ng = int(B.max().item()) + 1
        dense_h, mask = to_dense(h, B, ng)
        caps_out = self.caps(dense_h, mask)
        return self.head(caps_out.flatten(1))


def _load_warmstart(model):
    """Copy shape-matched encoder tensors from the multi-task GNN (§5), if it ran."""
    state = globals().get("MTL_ENCODER_STATE")
    if not state:
        return
    sd = model.state_dict()
    matched = {k: v for k, v in state.items() if k in sd and sd[k].shape == v.shape}
    sd.update(matched); model.load_state_dict(sd)


def _caps_predict(model, graphs, bs=256):
    model.eval(); out = np.zeros(len(graphs), np.float32)
    with torch.no_grad():
        for i in range(0, len(graphs), bs):
            X, EI, EA, B = collate(graphs[i:i + bs])
            X, EI, EA, B = X.to(DEVICE), EI.to(DEVICE), EA.to(DEVICE), B.to(DEVICE)
            out[i:i + bs] = model(X, EI, EA, B).squeeze(-1).cpu().numpy()
    return out


def train_capsnet_target(target, epochs=90, patience=18):
    is_small = target in SMALL_TARGETS
    lr_e = 1.5e-4 if is_small else 3e-4
    bs = 32 if is_small else 96
    mask = (train["target_type"].values == target)
    rows = np.where(mask)[0]
    y = train.loc[mask, "target"].values.astype(np.float32)
    groups = train.loc[mask, "smiles_canon"].values
    gsub = [G_tr[r] for r in rows]
    ok = [i for i, g in enumerate(gsub) if g is not None]
    if len(ok) < 0.9 * len(gsub):
        print(f"[{target}] only {len(ok)}/{len(gsub)} encodable -> skipping capsnet")
        return None

    oof = np.full(len(y), np.nan, np.float32)
    tst = np.zeros(len(test), np.float32); nfold = 0
    torch.manual_seed(SEED); np.random.seed(SEED)
    for tr_i, va_i in make_group_folds(groups, N_FOLDS, seed=SEED):
        if len(tr_i) < 20 or len(va_i) < 2:
            continue
        mu, sd = y[tr_i].mean(), y[tr_i].std() + 1e-8
        gtr = [gsub[i] for i in tr_i]; ytr = (y[tr_i] - mu) / sd
        gva = [gsub[i] for i in va_i]

        model = GNNCapsNet(ADIM, BDIM).to(DEVICE)
        _load_warmstart(model)
        if is_small:
            # ~220 rows can overwrite an inherited trunk in a couple of epochs -- freeze the
            # first two message-passing layers, same rationale as the restored heads in 5a.
            for p in model.lin0.parameters(): p.requires_grad_(False)
            for l in (0, 1):
                for p in model.msg[l].parameters(): p.requires_grad_(False)
                for p in model.upd[l].parameters(): p.requires_grad_(False)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                 lr=lr_e, weight_decay=1e-5)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

        idx = np.arange(len(gtr)); best = -1e9; best_state = None; wait = 0
        for ep in range(epochs):
            model.train(); np.random.shuffle(idx)
            for k in range(0, len(idx), bs):
                bi = idx[k:k + bs]
                if len(bi) < 2: continue
                X, EI, EA, B = collate([gtr[j] for j in bi])
                X, EI, EA, B = X.to(DEVICE), EI.to(DEVICE), EA.to(DEVICE), B.to(DEVICE)
                pred = model(X, EI, EA, B).squeeze(-1)
                tgt = torch.tensor(ytr[bi], device=DEVICE, dtype=torch.float32)
                loss = F.smooth_l1_loss(pred, tgt)
                opt.zero_grad(); loss.backward(); opt.step()
            sched.step()
            vp = _caps_predict(model, gva) * sd + mu
            r2 = r2_score(y[va_i], vp) if len(va_i) >= 5 else -1e9
            if r2 > best:
                best = r2; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}; wait = 0
            else:
                wait += 1
                if wait >= patience: break
        if best_state is None:
            continue
        model.load_state_dict(best_state)
        oof[va_i] = _caps_predict(model, gva) * sd + mu
        tst += (_caps_predict(model, G_te) * sd + mu) / N_FOLDS
        nfold += 1

    v = np.isfinite(oof)
    if nfold == 0 or v.sum() < 5:
        print(f"[{target}] too few valid folds -> skipping capsnet")
        return None
    print(f"[{target}] gnn-capsnet OOF R2 = {r2_score(y[v], oof[v]):.4f}  ({nfold}/{N_FOLDS} folds)")
    return {"oof": oof, "test": tst}


capsnet_res = {}
if GNNCAPS_ENABLE:
    print("Training GNN + CapsNet hybrid heads ...")
    for _t in CAPS_TARGETS:
        _r = train_capsnet_target(_t)
        if _r is not None:
            capsnet_res[_t] = _r


## 6. Meta-Stacking & Submission Generation

## 5c. Calibrated physics head — sibling Δ-learning

Diagnostic run on this training data (group-fold OOF, molecules where both values are known):

| identity | n | raw R² | **calibrated R²** |
|---|---|---|---|
| ei = eea + egc (Koopmans) | 59 | 0.963 | **0.965** |
| eea = ei − egc | 59 | 0.971 | **0.973** |
| egb ~ f(egc) | 175 | 0.892 | **0.928** |
| eps ~ f(nc²) (Maxwell) | 134 | 0.336 | **0.855** |
| nc ~ f(√eps) | 134 | 0.171 | **0.837** |

The raw identities are *quantitatively wrong but qualitatively right* — `eps = nc²` explains
almost nothing raw (0.336) yet 0.855 after a two-parameter rescale. The pipeline already feeds
the **uncalibrated** column as a feature, which asks a tree on ~220 rows to discover that
rescale from a single column. Fitting it explicitly with a regularized linear model is far more
sample-efficient — the standard small-data recipe of correcting a cheap approximate estimator
by linear regression calibrated on the target data.

**Applied as a gated blend, not an override.** The head only beats the existing stack outright
on ei/eea; for egb/eps/nc it is individually *worse*. So α is fitted per target by CV on covered
rows with 0 in the grid — a target that gains nothing keeps its stacked prediction unchanged.

Sibling values come from *other properties of the same molecule*, never the row's own label, and
train-side coverage is computed exactly as test-side is, so the OOF estimate matches deployment.

In [17]:
"""Calibrated physics head (delta-learning style sibling transfer).

Measured motivation (group-fold OOF on this exact training data, covered rows only):

    target  coverage   physics-head R2    existing stack R2
      ei      26.6%        0.9650              0.8801
     eea      26.7%        0.9728              0.9301
     egc       2.9%        0.9750              0.9252
     egb      51.9%        0.9273              0.9493
     eps      58.5%        0.8498              0.8610
      nc      58.5%        0.8398              0.9089

Why a *calibrated* head and not just the raw identity feature (which the pipeline already
has): the raw identities are quantitatively wrong but qualitatively right. On molecules where
both values are known, eps = nc^2 scores R2 = 0.336 raw but 0.855 after a two-parameter linear
rescale; nc = sqrt(eps) goes 0.171 -> 0.837. Koopmans is the exception, already near-exact at
0.963 raw. Feeding the uncalibrated column into a tree on ~220 rows asks the model to discover
that rescale from one feature; fitting it explicitly with a regularized linear model is far
more sample-efficient. This mirrors the standard small-data transfer recipe of correcting a
cheap approximate estimator with a linear regression calibrated on the target data.

Honesty about scope: the head is only individually better than the current stack on ei/eea
(and egc, which has almost no coverage). For egb/eps/nc it is individually *worse*, so it is
added as a GATED BLEND -- alpha is fitted per target on covered rows by cross-validation, and
alpha = 0 is allowed, meaning a target that gains nothing simply ignores this member. No
target can be made worse by it.

Leakage note: sibling values come from OTHER properties of the same molecule, never from the
row's own label. Train-side coverage is computed the same way test-side is (sibling must exist
in TRAIN), so the OOF estimate matches deployment conditions.
"""
from sklearn.linear_model import RidgeCV

PHYS_RECIPES = {
    "ei":  (["eea", "egc"], lambda a, b: a + b),          # Koopmans
    "eea": (["ei", "egc"],  lambda a, b: a - b),          # Koopmans
    "egc": (["ei", "eea"],  lambda a, b: a - b),          # Koopmans
    "egb": (["egc"],        lambda a: a),                 # bulk vs chain bandgap
    "eps": (["nc"],         lambda a: a ** 2),            # Maxwell / Clausius-Mossotti
    "nc":  (["eps"],        lambda a: np.sqrt(np.clip(a, 1e-6, None))),
}
PHYS_MIN_COVERED = 25          # below this, calibration is not trustworthy

# real (non-imputed) sibling lookup tables, built from TRAIN only
_PLUT = {t: dict(zip(train.loc[train["target_type"] == t, "smiles_canon"],
                     train.loc[train["target_type"] == t, "target"]))
         for t in TARGETS}


def _phys_design(mols, sibs, fn):
    """Return (design matrix, coverage mask). Design = [identity estimate, raw siblings]."""
    S = np.full((len(mols), len(sibs)), np.nan, dtype=np.float64)
    for j, s in enumerate(sibs):
        lut = _PLUT.get(s, {})
        S[:, j] = [lut.get(m, np.nan) for m in mols]
    cov = np.isfinite(S).all(axis=1)
    F = np.full((len(mols), len(sibs) + 1), np.nan, dtype=np.float64)
    if cov.any():
        cols = [S[cov, j] for j in range(len(sibs))]
        F[cov, 0] = fn(*cols)
        for j in range(len(sibs)):
            F[cov, j + 1] = cols[j]
    return F, cov


def train_physics_head(target):
    if target not in PHYS_RECIPES:
        return None
    sibs, fn = PHYS_RECIPES[target]
    if any(s not in TARGETS for s in sibs):
        return None
    mask = (train["target_type"].values == target)
    y = train.loc[mask, "target"].values.astype(np.float64)
    mols_tr = train.loc[mask, "smiles_canon"].values
    F_tr, cov_tr = _phys_design(mols_tr, sibs, fn)
    if cov_tr.sum() < PHYS_MIN_COVERED:
        print(f"[{target}] physics head: only {int(cov_tr.sum())} covered rows -> skipped")
        return None

    # honest OOF on covered rows, grouped by molecule
    oof = np.full(len(y), np.nan)
    Fc, yc, gc = F_tr[cov_tr], y[cov_tr], mols_tr[cov_tr]
    k = int(min(5, max(2, len(np.unique(gc)) // 10)))
    for a, b in make_group_folds(gc, k, seed=SEED):
        mdl = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0]).fit(Fc[a], yc[a])
        oof[np.where(cov_tr)[0][b]] = mdl.predict(Fc[b])
    r2_cov = r2_score(yc, oof[cov_tr])

    # refit on all covered training rows for the test prediction
    full = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0]).fit(Fc, yc)
    mols_te = test["smiles_canon"].values
    F_te, cov_te = _phys_design(mols_te, sibs, fn)
    tst = np.full(len(test), np.nan)
    if cov_te.any():
        tst[cov_te] = full.predict(F_te[cov_te])

    print(f"[{target}] physics head ({'+'.join(sibs)}): "
          f"train cov {int(cov_tr.sum())}/{len(y)} ({100*cov_tr.mean():.1f}%), "
          f"test cov {int(cov_te.sum())}/{len(test)} ({100*cov_te.mean():.1f}%), "
          f"OOF R2 on covered = {r2_cov:.4f}")
    return {"oof": oof, "cov_tr": cov_tr, "test": tst, "cov_te": cov_te, "r2_cov": r2_cov}


print("=" * 70)
print("Calibrated physics heads (sibling delta-learning)")
print("=" * 70)
phys_res = {}
for t in TARGETS:
    r = train_physics_head(t)
    if r is not None:
        phys_res[t] = r


def apply_physics_gate(target, stack_oof, stack_test, y, valid_mask):
    """Blend the physics head into an existing stacked prediction, on covered rows only.

    alpha is chosen by group-fold CV over the covered rows; alpha = 0 is in the grid, so a
    target that gains nothing from the physics head keeps its stacked prediction unchanged.
    Returns (new_oof, new_test, alpha, gain).
    """
    if target not in phys_res:
        return stack_oof, stack_test, 0.0, 0.0
    pr = phys_res[target]
    cov = pr["cov_tr"][valid_mask]
    if cov.sum() < PHYS_MIN_COVERED:
        return stack_oof, stack_test, 0.0, 0.0
    p_oof = pr["oof"][valid_mask]
    base_r2 = r2_score(y, stack_oof)

    grid = np.linspace(0.0, 1.0, 21)
    mols = train.loc[(train["target_type"].values == target), "smiles_canon"].values[valid_mask]
    gc = mols[cov]
    k = int(min(5, max(2, len(np.unique(gc)) // 10)))
    # pick alpha by CV *on covered rows*, scoring the full-vector R2 each time
    cv_scores = np.zeros(len(grid))
    for a, b in make_group_folds(gc, k, seed=SEED):
        ev = np.where(cov)[0][b]
        for gi, al in enumerate(grid):
            trial = stack_oof.copy()
            trial[ev] = (1 - al) * stack_oof[ev] + al * p_oof[ev]
            cv_scores[gi] += r2_score(y, trial)
    alpha = float(grid[int(np.argmax(cv_scores))])

    new_oof = stack_oof.copy()
    new_oof[cov] = (1 - alpha) * stack_oof[cov] + alpha * p_oof[cov]
    new_r2 = r2_score(y, new_oof)

    new_test = stack_test.copy()
    ct = pr["cov_te"] & np.isfinite(pr["test"])
    if alpha > 0 and ct.any():
        new_test[ct] = (1 - alpha) * stack_test[ct] + alpha * pr["test"][ct]
    gain = new_r2 - base_r2
    print(f"[{target}] physics gate: alpha={alpha:.2f}  OOF {base_r2:.4f} -> {new_r2:.4f} "
          f"({gain:+.4f})")
    return new_oof, new_test, alpha, gain


Calibrated physics heads (sibling delta-learning)
[eea] physics head (ei+egc): train cov 59/221 (26.7%), test cov 189/4940 (3.8%), OOF R2 on covered = 0.9716
[egb] physics head (egc): train cov 175/337 (51.9%), test cov 758/4940 (15.3%), OOF R2 on covered = 0.9259
[egc] physics head (ei+eea): train cov 59/2028 (2.9%), test cov 208/4940 (4.2%), OOF R2 on covered = 0.9740
[ei] physics head (eea+egc): train cov 59/222 (26.6%), test cov 188/4940 (3.8%), OOF R2 on covered = 0.9628
[eps] physics head (nc): train cov 134/229 (58.5%), test cov 449/4940 (9.1%), OOF R2 on covered = 0.8460
[nc] physics head (eps): train cov 134/229 (58.5%), test cov 443/4940 (9.0%), OOF R2 on covered = 0.8364


In [18]:
def _fit_predict(method, Mtr, ytr, Mev):
    if method == "mean":
        return Mev.mean(axis=1)
    if method == "nnls":
        w, _ = nnls(Mtr, ytr)
        if w.sum() == 0:
            w = np.ones(Mtr.shape[1])
        w = w / w.sum()
        return Mev @ w
    if method == "ridge":
        return Ridge(alpha=1.0, positive=True).fit(Mtr, ytr).predict(Mev)
    if method == "trim":
        # drop the single worst column then average the rest -- cheap robustness against one
        # member drifting on a particular fold, without fitting any weights
        if Mtr.shape[1] <= 2:
            return Mev.mean(axis=1)
        col_r2 = [r2_score(ytr, Mtr[:, j]) for j in range(Mtr.shape[1])]
        keep = [j for j in range(Mtr.shape[1]) if j != int(np.argmin(col_r2))]
        return Mev[:, keep].mean(axis=1)
    raise ValueError(method)


def stack(target):
    y = tree_res[target]["y"]
    base_mask = tree_res[target]["mask"]
    _tn = list(tree_res[target]["oof"].keys())

    base = {n: tree_res[target]["oof"][n] for n in _tn}
    test_base = {n: tree_res[target]["test"][n] for n in _tn}

    base["ridge"] = ridge_res[target]["oof"]
    test_base["ridge"] = ridge_res[target]["test"]
    base["mtl"] = mtl_res[target]["oof"]
    test_base["mtl"] = mtl_res[target]["test"]

    # --- new decorrelated members (added for every target where available) ---
    if target in gnn_res:
        base["gnn"] = gnn_res[target]["oof"]
        test_base["gnn"] = gnn_res[target]["test"]
    if target in gpr_res:
        base["gpr"] = gpr_res[target]["oof"]
        test_base["gpr"] = gpr_res[target]["test"]
    if target in nyskrr_res:
        base["nyskrr"] = nyskrr_res[target]["oof"]
        test_base["nyskrr"] = nyskrr_res[target]["test"]
    if target in et_res:
        base["et"] = et_res[target]["oof"]
        test_base["et"] = et_res[target]["test"]
    if target in capsnet_res:
        base["capsnet"] = capsnet_res[target]["oof"]
        test_base["capsnet"] = capsnet_res[target]["test"]

    names = list(base.keys())
    valid = np.ones(len(y), bool)
    for n in names:
        valid &= np.isfinite(base[n])
    yv = y[valid]

    _r2 = {n: r2_score(yv, base[n][valid]) for n in names}
    _best = max(_r2.values())
    kept = [n for n in names if _r2[n] > max(0.0, _best - 0.10)]
    if len(kept) < 1:
        kept = [max(_r2, key=_r2.get)]
    dropped = [n for n in names if n not in kept]
    for n in sorted(names, key=lambda z: -_r2[z]):
        flag = "" if n in kept else "   (pruned)"
        print(f"[{target}] {n:>8} OOF R2 = {_r2[n]:.4f}{flag}")
    names = kept
    M = np.column_stack([base[n][valid] for n in names])

    # `single` is included as a candidate so the meta-learner can decline to blend when a
    # lone member genuinely dominates -- on the smallest targets fitting weights on ~220
    # points can be worse than just taking the best base model.
    best_single = max(names, key=lambda n: _r2[n])
    methods = ["mean", "nnls", "ridge", "trim", "single"]
    _mg = train.loc[base_mask, "smiles_canon"].values[valid]
    cv_pred = {m: np.zeros(len(yv)) for m in methods}
    for tr, ev in make_group_folds(_mg, 5, seed=SEED):
        for m in methods:
            if m == "single":
                cv_pred[m][ev] = base[best_single][valid][ev]
            else:
                cv_pred[m][ev] = _fit_predict(m, M[tr], yv[tr], M[ev])
    cv_r2 = {m: r2_score(yv, cv_pred[m]) for m in methods}
    choice = max(cv_r2, key=cv_r2.get)
    print(f"[{target}] meta-CV: " + " ".join(f"{m}={cv_r2[m]:.4f}" for m in methods)
          + f"  -> {choice}")

    Tm = np.column_stack([test_base[n] for n in names])
    if choice == "single":
        test_pred = test_base[best_single]
    else:
        test_pred = _fit_predict(choice, M, yv, Tm)

    # --- calibrated physics head, applied as a gated blend on covered rows only ---
    # alpha is CV-selected per target with 0 in the grid, so this can only help or no-op.
    stack_oof_full = cv_pred[choice]
    new_oof, test_pred, _alpha, _gain = apply_physics_gate(
        target, stack_oof_full, test_pred, yv, valid)
    final_r2 = r2_score(yv, new_oof)
    return final_r2, test_pred, new_oof, valid


final = {}; oof_scores = {}; stack_oof = {}; stack_valid = {}
for t in TARGETS:
    r2, tp, _so, _vv = stack(t)
    final[t] = tp; oof_scores[t] = r2
    stack_oof[t] = _so; stack_valid[t] = _vv
    print()

mean_oof = float(np.mean(list(oof_scores.values())))
print("HONEST FINAL OOF (nested, group-fold meta-CV) per target:")
for t in TARGETS:
    print(f"   {t:>4}: {oof_scores[t]:.4f}")
print(f"   MEAN R2 (competition metric) = {mean_oof:.4f}")

# ══════════════════════════════════════════════════════════════════════
# Graph-transformer blend — FIXED weight, applied after stacking
# ══════════════════════════════════════════════════════════════════════
# The transformer is NOT added to base/test_base, because doing so makes the
# meta-learner re-solve its weights on ~220 OOF rows, and re-solving is exactly
# what is unstable here: two pipelines with near-identical OOF (0.9099, 0.9114)
# scored 0.901 and 0.894. So instead the stack is left completely alone and the
# transformer is mixed in afterwards at a fixed weight.
#
# GT_BLEND_W = 0.0  ->  this notebook is byte-identical to the 0.903 run.
GT_BLEND_W = 0.15
# gt measured OOF: eea .9275  egb .9200  egc .9190  tg .9104 | ei .8319  nc .8466  eps .7694
GT_BLEND_TARGETS = ["eea", "egb", "egc", "tg"]

_gt = globals().get("gt_res", {})
if GT_BLEND_W > 0 and _gt:
    print("=" * 70)
    print(f"Graph-transformer blend at fixed w={GT_BLEND_W} on {GT_BLEND_TARGETS}")
    print("=" * 70)
    _new_scores = dict(oof_scores)
    for t in GT_BLEND_TARGETS:
        if t not in _gt: 
            print(f"  {t:5s} gt absent -> unchanged"); continue
        _m = (train["target_type"].values == t)
        _y = train.loc[_m, "target"].values.astype(float)
        _v = stack_valid[t]
        _o0 = stack_oof[t]
        _o1 = (1 - GT_BLEND_W) * _o0 + GT_BLEND_W * _gt[t]["oof"][_v]
        _r0, _r1 = r2_score(_y[_v], _o0), r2_score(_y[_v], _o1)
        final[t] = (1 - GT_BLEND_W) * final[t] + GT_BLEND_W * _gt[t]["test"]
        _new_scores[t] = _r1
        print(f"  {t:5s} stack {_r0:.4f} -> blended {_r1:.4f}  ({_r1-_r0:+.4f})")
    print(f"\n  MEAN OOF: {np.mean(list(oof_scores.values())):.4f} -> "
          f"{np.mean(list(_new_scores.values())):.4f}")
    print("  (OOF has not predicted the leaderboard here - treat this as a")
    print("   sanity check that the blend is not destructive, not as a forecast.)")
    oof_scores = _new_scores

pred = np.full(len(test), np.nan)
for t in TARGETS:
    m = (test["target_type"].values == t)
    pred[m] = final[t][m]
# safety net only: any non-finite prediction falls back to that target's training median
for t in TARGETS:
    m = (test["target_type"].values == t) & ~np.isfinite(pred)
    if m.any():
        pred[m] = train.loc[train["target_type"] == t, "target"].median()
        print(f"WARNING: filled {int(m.sum())} non-finite predictions for {t}")

# NOTE: the exact-match train->test override that previous versions applied here has been
# REMOVED. Copying ground-truth labels into the submission for molecules that happen to
# appear in both splits is answer-key retrieval rather than prediction; it is the pattern
# that has drawn disqualifications in this competition, and it was worth ~0.000 anyway
# (it matched only 2 of 4940 rows). Every value below is a model prediction.

sub = pd.DataFrame({"id": test["id"].values, "target": pred})
sub.to_csv("submission.csv", index=False)
print(f"\nSaved submission.csv {sub.shape}")
print(sub.head().to_string(index=False))


[eea]      gnn OOF R2 = 0.9453
[eea]      mtl OOF R2 = 0.9250
[eea]      cat OOF R2 = 0.9185
[eea]      lgb OOF R2 = 0.9131
[eea]      gpr OOF R2 = 0.9071
[eea]       et OOF R2 = 0.8953
[eea]    ridge OOF R2 = 0.8929
[eea]   nyskrr OOF R2 = 0.8362   (pruned)
[eea] meta-CV: mean=0.9368 nnls=0.9463 ridge=0.9463 trim=0.9371 single=0.9453  -> nnls
[eea] physics gate: alpha=0.65  OOF 0.9463 -> 0.9513 (+0.0049)

[egb]      mtl OOF R2 = 0.9465
[egb]      gnn OOF R2 = 0.9418
[egb]      lgb OOF R2 = 0.9399
[egb]      cat OOF R2 = 0.9383
[egb]      gpr OOF R2 = 0.9301
[egb]       et OOF R2 = 0.9295
[egb]    ridge OOF R2 = 0.8883
[egb]   nyskrr OOF R2 = 0.8768
[egb] meta-CV: mean=0.9453 nnls=0.9514 ridge=0.9522 trim=0.9487 single=0.9465  -> ridge
[egb] physics gate: alpha=0.20  OOF 0.9522 -> 0.9538 (+0.0016)

[egc]      gnn OOF R2 = 0.9292
[egc]      mtl OOF R2 = 0.9220
[egc]   lgb_et OOF R2 = 0.9120
[egc]      cat OOF R2 = 0.9105
[egc]      lgb OOF R2 = 0.9097
[egc]      xgb OOF R2 = 0.9096
[egc